# Imports:

In [43]:
import os
import random
import math
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import re
from peft import LoraConfig, get_peft_model
from peft import prepare_model_for_kbit_training
from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer
import time
import inspect
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments

C:\Users\acer\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Reproducibility:

In [44]:
torch.manual_seed(0)
random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


### Config:

In [45]:
SEED = 0
random.seed(SEED)

MAX_DIGITS = 4
OPS = ("+", "-")

N_TRAIN = 200_000     
N_VALID = 10_000      
N_RL    = 50_000      
N_TEST_SEEN          = 20_000
N_TEST_UNSEEN_NUMS   = 20_000
N_TEST_UNSEEN_FORMAT = 20_000

OUT_DIR = "data_task1"
os.makedirs(OUT_DIR, exist_ok=True)

Digit-length sampling weights (1..MAX_DIGITS) to avoid only short numbers dominating

In [46]:
DIGIT_WEIGHTS = {1: 0.20, 2: 0.30, 3: 0.30, 4: 0.20}

Training format(s): keep simple for scratch pretrain

In [47]:
SEEN_FORMATS = [
    "{a} {op} {b} ="
]

Unseen-format evaluation: ONLY use chars already in your vocab (digits, +, -, =, space)

In [48]:
UNSEEN_FORMATS = [
    "{a}{op}{b}=",
    "{a} {op}{b} =",
    "{a}{op} {b} =",
    "{a}  {op}  {b} =",   # double spaces
    " {a} {op} {b} =",    # leading space
    "{a} {op} {b} = "     # trailing space
]

### Helper Functions:

In [49]:
def sample_digit_len(max_digits=4):
    r = random.random()
    cum = 0.0
    for d in range(1, max_digits + 1):
        cum += DIGIT_WEIGHTS.get(d, 1.0 / max_digits)
        if r <= cum:
            return d
    return max_digits

In [50]:
def number_range_for_digits(d: int):
    if d == 1:
        return 0, 9
    lo = 10 ** (d - 1)
    hi = (10 ** d) - 1
    return lo, hi

In [51]:
def number_range_for_digits(d: int):
    if d == 1:
        return 0, 9
    lo = 10 ** (d - 1)
    hi = (10 ** d) - 1
    return lo, hi

In [52]:
def make_holdout_numbers_per_digit(max_digits=4, holdout_frac=0.10):
    """
    Build a held-out operand set per digit length so that "unseen numbers"
    test is truly unseen (no operand appears in training).
    """
    holdout = {}
    rng = random.Random(SEED + 12345)

    for d in range(1, max_digits + 1):
        lo, hi = number_range_for_digits(d)
        all_nums = list(range(lo, hi + 1))

        k = max(2 if d == 1 else 10, int(len(all_nums) * holdout_frac))
        k = min(k, len(all_nums) // 2)  

        holdout_set = set(rng.sample(all_nums, k=k))
        holdout[d] = holdout_set

    return holdout

In [53]:
HOLDOUT_BY_DIGIT = make_holdout_numbers_per_digit(MAX_DIGITS, holdout_frac=0.10)

In [54]:
def sample_number(d: int, mode: str):
    """
    mode:
      - "train": sample excluding holdout numbers for that digit
      - "holdout": sample only from holdout numbers for that digit
    """
    lo, hi = number_range_for_digits(d)
    if mode == "holdout":
        pool = list(HOLDOUT_BY_DIGIT[d])
        return random.choice(pool)

    # train mode
    # rejection sample until not in holdout (fast because holdout fraction small)
    while True:
        x = random.randint(lo, hi)
        if x not in HOLDOUT_BY_DIGIT[d]:
            return x

In [55]:
def compute_answer(a: int, b: int, op: str):
    if op == "+":
        return a + b
    # subtraction: enforce non-negative
    if a < b:
        a, b = b, a
    return a - b

In [56]:
def make_one_example(fmt_list, operand_mode="train"):
    op = random.choice(list(OPS))
    da = sample_digit_len(MAX_DIGITS)
    db = sample_digit_len(MAX_DIGITS)

    a = sample_number(da, mode=operand_mode)
    b = sample_number(db, mode=operand_mode)

    # For subtraction: keep non-negative
    if op == "-" and a < b:
        a, b = b, a

    ans = compute_answer(a, b, op)

    fmt_id = random.randrange(len(fmt_list))
    prompt = fmt_list[fmt_id].format(a=a, b=b, op=op)

    return {
        "prompt": prompt,
        "answer": str(ans),
        "op": op,
        "a": a,
        "b": b,
        "fmt_id": fmt_id,
        "len_a": len(str(a)),
        "len_b": len(str(b)),
        "len_ans": len(str(ans)),
    }

In [57]:
def generate_dataframe(n, fmt_list, operand_mode="train"):
    rows = [make_one_example(fmt_list, operand_mode=operand_mode) for _ in range(n)]
    return pd.DataFrame(rows)

### Generate splits:

In [58]:
df_train = generate_dataframe(N_TRAIN, SEEN_FORMATS, operand_mode="train")
df_valid = generate_dataframe(N_VALID, SEEN_FORMATS, operand_mode="train")

In [59]:
df_rl = generate_dataframe(N_RL, SEEN_FORMATS, operand_mode="train")

In [60]:
df_test_seen = generate_dataframe(N_TEST_SEEN, SEEN_FORMATS, operand_mode="train")

In [61]:
df_test_unseen_numbers = generate_dataframe(N_TEST_UNSEEN_NUMS, SEEN_FORMATS, operand_mode="holdout")

In [62]:
df_test_unseen_formats = generate_dataframe(N_TEST_UNSEEN_FORMAT, UNSEEN_FORMATS, operand_mode="train")

### Sanity checks:

In [63]:
train_operands = set(df_train["a"].tolist()) | set(df_train["b"].tolist())
unseen_operands = set(df_test_unseen_numbers["a"].tolist()) | set(df_test_unseen_numbers["b"].tolist())
overlap = len(train_operands & unseen_operands)

print("\nCounts:")
print("Train:", len(df_train))
print("Valid:", len(df_valid))
print("RL prompts:", len(df_rl))
print("Test seen:", len(df_test_seen))
print("Test unseen numbers:", len(df_test_unseen_numbers))
print("Test unseen formats:", len(df_test_unseen_formats))


Counts:
Train: 200000
Valid: 10000
RL prompts: 50000
Test seen: 20000
Test unseen numbers: 20000
Test unseen formats: 20000


In [64]:
print("\nOperand overlap check (train vs unseen_numbers):")
print("Overlap =", overlap)


Operand overlap check (train vs unseen_numbers):
Overlap = 0


In [65]:
print("\nOp balance (train):")
print(df_train["op"].value_counts(normalize=True))


Op balance (train):
op
-    0.500815
+    0.499185
Name: proportion, dtype: float64


In [66]:
print("\nDigit length balance (train) (len_a):")
print(df_train["len_a"].value_counts(normalize=True).sort_index())


Digit length balance (train) (len_a):
len_a
1    0.119180
2    0.256045
3    0.344145
4    0.280630
Name: proportion, dtype: float64


### Save CSVs:

In [67]:
df_train.to_csv(os.path.join(OUT_DIR, "pretraining_train.csv"), index=False)
df_valid.to_csv(os.path.join(OUT_DIR, "pretraining_valid.csv"), index=False)
df_rl.to_csv(os.path.join(OUT_DIR, "rl_prompts.csv"), index=False)
df_test_seen.to_csv(os.path.join(OUT_DIR, "test_seen.csv"), index=False)
df_test_unseen_numbers.to_csv(os.path.join(OUT_DIR, "test_unseen_numbers.csv"), index=False)
df_test_unseen_formats.to_csv(os.path.join(OUT_DIR, "test_unseen_formats.csv"), index=False)

print("\nSaved files to:", OUT_DIR)


Saved files to: data_task1


Preview a few examples

In [68]:
df_train.head(5)

,prompt,answer,op,a,b,fmt_id,len_a,len_b,len_ans
0,365 - 75 =,290,-,365,75,0,3,2,3
1,8808 - 6866 =,1942,-,8808,6866,0,4,4,4
2,89 + 918 =,1007,+,89,918,0,2,3,4
3,97 + 438 =,535,+,97,438,0,2,3,3
4,91 + 36 =,127,+,91,36,0,2,2,3


In [69]:
df_test_unseen_numbers.head(5)

,prompt,answer,op,a,b,fmt_id,len_a,len_b,len_ans
0,43 - 0 =,43,-,43,0,0,2,1,2
1,636 - 222 =,414,-,636,222,0,3,3,3
2,127 + 850 =,977,+,127,850,0,3,3,3
3,43 + 6 =,49,+,43,6,0,2,1,2
4,408 + 44 =,452,+,408,44,0,3,2,3


In [70]:
df_test_unseen_formats.head(5)

,prompt,answer,op,a,b,fmt_id,len_a,len_b,len_ans
0,965-4=,961,-,965,4,0,3,1,3
1,2936-45=,2891,-,2936,45,0,4,2,4
2,84+ 71 =,155,+,84,71,2,2,2,3
3,22 +4 =,26,+,22,4,1,2,1,2
4,72+902=,974,+,72,902,0,2,3,3


### Load CSV splits:

In [71]:
SEED = 0
random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = "data_task1" 

In [72]:
train_path = os.path.join(DATA_DIR, "pretraining_train.csv")
valid_path = os.path.join(DATA_DIR, "pretraining_valid.csv")
rl_path    = os.path.join(DATA_DIR, "rl_prompts.csv")

In [73]:
test_seen_path          = os.path.join(DATA_DIR, "test_seen.csv")
test_unseen_numbers_path= os.path.join(DATA_DIR, "test_unseen_numbers.csv")
test_unseen_formats_path= os.path.join(DATA_DIR, "test_unseen_formats.csv")

In [74]:
assert os.path.exists(train_path), f"Missing: {train_path}"
assert os.path.exists(valid_path), f"Missing: {valid_path}"
assert os.path.exists(rl_path),    f"Missing: {rl_path}"

In [75]:
df_train = pd.read_csv(train_path)
df_valid = pd.read_csv(valid_path)
df_rl    = pd.read_csv(rl_path)

In [76]:
df_test_seen = pd.read_csv(test_seen_path) if os.path.exists(test_seen_path) else None
df_test_unseen_numbers = pd.read_csv(test_unseen_numbers_path) if os.path.exists(test_unseen_numbers_path) else None
df_test_unseen_formats = pd.read_csv(test_unseen_formats_path) if os.path.exists(test_unseen_formats_path) else None

In [77]:
print("Train:", len(df_train), "Valid:", len(df_valid), "RL prompts:", len(df_rl))

Train: 200000 Valid: 10000 RL prompts: 50000


In [78]:
df_train.head()

,prompt,answer,op,a,b,fmt_id,len_a,len_b,len_ans
0,365 - 75 =,290,-,365,75,0,3,2,3
1,8808 - 6866 =,1942,-,8808,6866,0,4,4,4
2,89 + 918 =,1007,+,89,918,0,2,3,4
3,97 + 438 =,535,+,97,438,0,2,3,3
4,91 + 36 =,127,+,91,36,0,2,2,3


### Vocab:
Character-level vocab

In [79]:
vocab = ['<pad>', '<bos>', '<eos>'] + ['0','1','2','3','4','5','6','7','8','9', '+', '-', '=', ' ']
vocab2id = {ch: i for i, ch in enumerate(vocab)}
id2vocab = {i: ch for i, ch in enumerate(vocab)}

PAD_ID = vocab2id['<pad>']
BOS_ID = vocab2id['<bos>']
EOS_ID = vocab2id['<eos>']
vocab_size = len(vocab)

print("Vocab size:", vocab_size)
print("PAD/BOS/EOS:", PAD_ID, BOS_ID, EOS_ID)

Vocab size: 17
PAD/BOS/EOS: 0 1 2


### encode/decode:

In [80]:
def encode(text: str):
    for c in text:
        if c not in vocab2id:
            raise ValueError(f"Character {repr(c)} not in vocab.")
    return [BOS_ID] + [vocab2id[c] for c in text] + [EOS_ID]

In [81]:
def decode(ids):
    out = []
    for i in ids:
        tok = id2vocab[int(i)]
        if tok in ('<pad>','<bos>','<eos>'):
            continue
        out.append(tok)
    return ''.join(out)

quick sanity check

In [82]:
s = "103 - 88 ="
print("Encoded:", encode(s)[:10])
print("Decoded:", decode(encode(s)))

Encoded: [1, 4, 3, 6, 16, 14, 16, 11, 11, 16]
Decoded: 103 - 88 =


### Dataset that masks prompt loss:

In [83]:
def normalize_prompt_eq_space(p: str) -> str:
    """
    Canonical format for BOTH training and generation:
    - strip
    - ensure it ends with '= ' (equal sign then exactly one space)
    Examples:
      '12 + 7 ='   -> '12 + 7 = '
      '12 + 7 ='   -> '12 + 7 = '
      '12+7='      -> '12+7= '  (still OK; uses same chars)
    """
    p = str(p).strip()

    if '=' not in p:
        p = p + " ="

    p = p.rstrip()

    if p.endswith('='):
        return p + ' '
    if p.endswith('= '):
        return p
    if p.rstrip().endswith('='):
        return p.rstrip() + ' '
    return p + ' = '

In [84]:
class MathLMPretrainDatasetMasked(Dataset):
    """
    Train a causal LM on: full_text = prompt_norm + answer
    where prompt_norm ends with '= '.
    Loss is masked on the prompt tokens, so we only learn the answer digits (+ EOS).
    """
    def __init__(self, df):
        self.prompts = df["prompt"].tolist()
        self.answers = df["answer"].tolist()

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, idx):
        prompt = normalize_prompt_eq_space(self.prompts[idx])
        answer = str(self.answers[idx]).strip()

        full_text = prompt + answer  

        ids = encode(full_text)
        x = torch.tensor(ids[:-1], dtype=torch.long)
        y = torch.tensor(ids[1:], dtype=torch.long)

        L = len(prompt)  
        y[:L] = PAD_ID   

        return x, y

### Collate/padding:

In [85]:
def collate_batch(batch):
    xs, ys = zip(*batch)
    max_len = max(x.size(0) for x in xs)
    x_pad, y_pad = [], []
    for x, y in zip(xs, ys):
        pad_len = max_len - x.size(0)
        x_pad.append(torch.cat([x, torch.full((pad_len,), PAD_ID, dtype=torch.long)]))
        y_pad.append(torch.cat([y, torch.full((pad_len,), PAD_ID, dtype=torch.long)]))
    return torch.stack(x_pad), torch.stack(y_pad)

### DataLoaders:

In [86]:
train_loader = DataLoader(MathLMPretrainDatasetMasked(df_train), batch_size=64, shuffle=True, collate_fn=collate_batch)
valid_loader = DataLoader(MathLMPretrainDatasetMasked(df_valid), batch_size=64, shuffle=False, collate_fn=collate_batch)

sanity check: first unmasked label should be first answer digit

In [87]:
x0, y0 = MathLMPretrainDatasetMasked(df_train)[0]
print("Decoded x:", decode(x0.tolist()))
print("Decoded y (unmasked):", decode([t for t in y0.tolist() if t != PAD_ID]))

Decoded x: 365 - 75 = 290
Decoded y (unmasked): 290


show where masking switches from PAD to real targets

In [88]:
pad_prefix = 0
for t in y0.tolist():
    if t == PAD_ID:
        pad_prefix += 1
    else:
        break
print("Masked target length (should equal len(prompt+' ')):", pad_prefix)

Masked target length (should equal len(prompt+' ')): 11


compute max length to set model.max_len later

In [89]:
def max_seq_len(df):
    mx = 0
    for _, r in df.iterrows():
        prompt = normalize_prompt_eq_space(r["prompt"])
        answer = str(r["answer"]).strip()
        full_text = prompt + " " + answer
        mx = max(mx, len(encode(full_text)) - 1)  
    return mx

In [90]:
mx_train = max_seq_len(df_train.sample(5000, random_state=SEED))  

print("Estimated max x length (from 5k samples):", mx_train)

Estimated max x length (from 5k samples): 21


batch shape sanity

In [91]:
xb, yb = next(iter(train_loader))

print("Batch shapes:", xb.shape, yb.shape)

Batch shapes: torch.Size([64, 18]) torch.Size([64, 18])


## Experiment (1) — From scratch (TinyTransformer):

In [92]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=128):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)
        self.register_buffer("pe", pe)  

    def forward(self, x):

        return x + self.pe[:x.size(1)]

In [93]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=256, num_heads=4, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.h = num_heads
        self.dk = d_model // num_heads
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.out = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # x: (B, T, C)
        B, T, C = x.size()
        q = self.q(x).view(B, T, self.h, self.dk).transpose(1, 2)  
        k = self.k(x).view(B, T, self.h, self.dk).transpose(1, 2)
        v = self.v(x).view(B, T, self.h, self.dk).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.dk)      
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = attn @ v                                               
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out(out)

In [94]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model=256, heads=4, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, heads, dropout=dropout)
        self.dropout1 = nn.Dropout(dropout)

        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, mlp_ratio * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_ratio * d_model, d_model),
        )
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = x + self.dropout1(self.attn(self.ln1(x), mask))
        x = x + self.dropout2(self.mlp(self.ln2(x)))
        return x

In [95]:
class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_layers=4, heads=4, max_len=128, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model, max_len=max_len)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, heads, mlp_ratio=4, dropout=dropout) for _ in range(n_layers)
        ])
        self.ln = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, vocab_size)
        self.max_len = max_len

    def forward(self, x):
        # x: (B, T)
        B, T = x.size()
        if T > self.max_len:
            raise ValueError(f"Sequence length T={T} exceeds max_len={self.max_len}.")

        mask = torch.tril(torch.ones(T, T, device=x.device))  

        h = self.embed(x)     
        h = self.pos(h)
        for blk in self.blocks:
            h = blk(h, mask)
        h = self.ln(h)
        logits = self.fc(h)   
        return logits

### Instantiate + training utilities:

In [96]:
model = TinyTransformer(vocab_size=vocab_size, d_model=256, n_layers=4, heads=4, max_len=128, dropout=0.1).to(device)

print("Params (M):", sum(p.numel() for p in model.parameters())/1e6)

Params (M): 3.168273


In [97]:
os.makedirs("checkpoints", exist_ok=True)

In [98]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=0.01)

In [99]:
warmup_steps = 2000
def lr_lambda(step): return min((step+1)/warmup_steps, 1.0)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

use_amp = (device == "cuda")
scaler = GradScaler(enabled=use_amp)

C:\Users\acer\AppData\Local\Temp\ipykernel_14468\3826650980.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=use_amp)


In [100]:
def train_one_epoch(model, loader):
    model.train()
    total = 0.0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=use_amp):
            logits = model(x)
            B,T,V = logits.shape
            loss = criterion(logits.view(B*T, V), y.view(B*T))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total += float(loss.item())
    return total/len(loader)

In [101]:
@torch.no_grad()
def eval_loss(model, loader):
    model.eval()
    total = 0.0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        with autocast(enabled=use_amp):
            logits = model(x)
            B,T,V = logits.shape
            loss = criterion(logits.view(B*T, V), y.view(B*T))
        total += float(loss.item())
    return total/len(loader)

### Greedy generation & exact-match accuracy:

In [102]:
DIGIT_IDS = [vocab2id[str(d)] for d in range(10)]
ALLOWED_IDS = set(DIGIT_IDS + [EOS_ID])

In [103]:
@torch.no_grad()
def generate_answer_scratch(model, prompt, max_answer_len=8):
    model.eval()
    prompt = normalize_prompt_eq_space(prompt)
    ids = encode(prompt)
    input_ids = torch.tensor(ids[:-1], dtype=torch.long).unsqueeze(0).to(device)

    for _ in range(max_answer_len):
        logits = model(input_ids)[0, -1, :]
        masked = torch.full_like(logits, float("-inf"))
        for i in ALLOWED_IDS:
            masked[i] = logits[i]
        next_id = int(torch.argmax(masked).item())
        if next_id == EOS_ID:
            break
        input_ids = torch.cat([input_ids, torch.tensor([[next_id]], device=device)], dim=1)

    gen = input_ids[0].tolist()
    return decode(gen[len(ids[:-1]):]).strip()

In [104]:
@torch.no_grad()
def exact_match_accuracy_scratch(model, df, max_items=1000, seed=0):
    df = df.sample(max_items, random_state=seed).reset_index(drop=True)
    correct = 0
    correct_add = correct_sub = 0
    total_add = total_sub = 0
    buckets, buckets_tot = {}, {}

    for _, r in df.iterrows():
        gt = str(r["answer"]).strip()
        pred = generate_answer_scratch(model, r["prompt"], max_answer_len=8)
        ok = (pred == gt)
        correct += int(ok)

        if r["op"] == "+":
            total_add += 1
            correct_add += int(ok)
        else:
            total_sub += 1
            correct_sub += int(ok)

        k = int(r["len_a"])
        buckets_tot[k] = buckets_tot.get(k, 0) + 1
        buckets[k] = buckets.get(k, 0) + int(ok)

    overall = correct / len(df)
    add_acc = correct_add / max(total_add, 1)
    sub_acc = correct_sub / max(total_sub, 1)
    by_len_a = {k: buckets[k] / buckets_tot[k] for k in sorted(buckets_tot)}
    return overall, add_acc, sub_acc, by_len_a

### Training:

In [133]:
best_em = -1.0
best_val = float("inf")

n_epochs = 50  
for epoch in range(1, n_epochs + 1):
    tr = train_one_epoch(model, train_loader)
    va = eval_loss(model, valid_loader)

    overall, add_acc, sub_acc, by_len = exact_match_accuracy_scratch(model, df_valid, max_items=1000, seed=0)

    print(f"Epoch {epoch:02d}: train_loss={tr:.4f} val_loss={va:.4f}")
    print(f"  EM@1000 valid: overall={overall:.3f} add={add_acc:.3f} sub={sub_acc:.3f} by_len_a={by_len}")

    torch.save(model.state_dict(), "checkpoints/sup_scratch_last.pt")

    if overall > best_em:
        best_em = overall
        torch.save(model.state_dict(), "checkpoints/sup_scratch_best_by_em.pt")
        print("  >>> Saved new BEST (by EM)")

    if va < best_val:
        best_val = va
        torch.save(model.state_dict(), "checkpoints/sup_scratch_best_by_loss.pt")

C:\Users\acer\AppData\Local\Temp\ipykernel_13728\2425358127.py:8: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


Epoch 01: train_loss=1.4427 val_loss=0.7467
  EM@1000 valid: overall=0.306 add=0.279 sub=0.334 by_len_a={1: 0.5306122448979592, 2: 0.37349397590361444, 3: 0.25925925925925924, 4: 0.18214285714285713}
  >>> Saved new BEST (by EM)
Epoch 02: train_loss=0.5234 val_loss=0.1996
  EM@1000 valid: overall=0.732 add=0.777 sub=0.686 by_len_a={1: 0.9727891156462585, 2: 0.8273092369477911, 3: 0.6944444444444444, 4: 0.5642857142857143}
  >>> Saved new BEST (by EM)
Epoch 03: train_loss=0.2650 val_loss=0.1238
  EM@1000 valid: overall=0.848 add=0.860 sub=0.836 by_len_a={1: 0.9659863945578231, 2: 0.9076305220883534, 3: 0.8209876543209876, 4: 0.7642857142857142}
  >>> Saved new BEST (by EM)
Epoch 04: train_loss=0.1914 val_loss=0.0913
  EM@1000 valid: overall=0.904 add=0.921 sub=0.887 by_len_a={1: 0.9863945578231292, 2: 0.9558232931726908, 3: 0.8765432098765432, 4: 0.8464285714285714}
  >>> Saved new BEST (by EM)
Epoch 05: train_loss=0.1467 val_loss=0.0600
  EM@1000 valid: overall=0.934 add=0.947 sub=0.92

### Evaluate on our test splits:

Let's load the best checkpoint:

In [77]:
model.load_state_dict(torch.load("checkpoints/sup_scratch_best.pt", map_location=device))

C:\Users\acer\AppData\Local\Temp\ipykernel_7504\1262682876.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("checkpoints/sup_scratch_best

<All keys matched successfully>

In [83]:
def report_split_fixed(name, df, max_items=2000):
    
    overall, add_acc, sub_acc, by_len = exact_match_accuracy_scratch(model, df, max_items=max_items, seed=0)
    print(f"{name}: overall={overall:.4f} add={add_acc:.4f} sub={sub_acc:.4f} by_len_a={by_len}")

In [79]:
model.load_state_dict(torch.load("checkpoints/sup_scratch_best_by_em.pt", map_location=device))
model.eval()

report_split_fixed("valid", df_valid, max_items=2000)
report_split_fixed("test_seen", df_test_seen, max_items=5000)
report_split_fixed("test_unseen_numbers", df_test_unseen_numbers, max_items=5000)
report_split_fixed("test_unseen_formats", df_test_unseen_formats, max_items=5000)

C:\Users\acer\AppData\Local\Temp\ipykernel_7504\3755294972.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("checkpoints/sup_scratch_best

valid: overall=0.9960 add=0.9980 sub=0.9939 by_len_a={1: 1.0, 2: 1.0, 3: 0.9957142857142857, 4: 0.9909420289855072}
test_seen: overall=0.9956 add=0.9980 sub=0.9932 by_len_a={1: 1.0, 2: 1.0, 3: 0.9966139954853274, 4: 0.9886363636363636}
test_unseen_numbers: overall=0.9936 add=0.9973 sub=0.9898 by_len_a={1: 1.0, 2: 0.9903632320237212, 3: 0.994001199760048, 4: 0.9934258582907232}
test_unseen_formats: overall=0.3736 add=0.3598 sub=0.3880 by_len_a={1: 0.46611570247933887, 2: 0.33490566037735847, 3: 0.3854587420657819, 4: 0.3539568345323741}


In [81]:
for i in range(15):
    r = df_valid.iloc[i]
    p = r["prompt"]
    gt = str(r["answer"]).strip()
    pred = generate_answer_scratch(model, p, max_answer_len=8)
    print(f"{p}  gt={gt}  pred={pred}")

2804 - 51 =  gt=2753  pred=2753
89 + 47 =  gt=136  pred=136
5 + 5 =  gt=10  pred=10
62 + 4575 =  gt=4637  pred=4637
885 - 3 =  gt=882  pred=882
35 + 7 =  gt=42  pred=42
42 + 669 =  gt=711  pred=711
8968 - 425 =  gt=8543  pred=8543
537 - 382 =  gt=155  pred=155
188 + 1040 =  gt=1228  pred=1228
1486 - 79 =  gt=1407  pred=1407
972 - 480 =  gt=492  pred=492
982 + 29 =  gt=1011  pred=1011
501 - 89 =  gt=412  pred=412
2211 - 20 =  gt=2191  pred=2191


## Experiment (2) - Qwen2.5-0.5B-Instruct + QLoRA:

### Load Qwen model:

In [121]:
!pip -q install -U transformers accelerate peft
!pip -q install -U bitsandbytes


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\acer\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\acer\AppData\Local\Programs\Python\Python312\python.exe -m pip install --upgrade pip


In [117]:
len(df_train)

200000

In [118]:
len(df_valid)

10000

In [119]:
SEED = 0
random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [120]:
try:
    from accelerate.state import AcceleratorState
    AcceleratorState._reset_state()
    print("Accelerate state reset ✅")
except Exception as e:
    print("Accelerate reset skipped:", str(e)[:120])

if device == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

Accelerate state reset ✅


In [122]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [123]:
use_4bit = False

try:
    from transformers import BitsAndBytesConfig
    import bitsandbytes as bnb  

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    model_qwen = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map=None,
    )
    model_qwen = model_qwen.to("cuda")
    use_4bit = True
    print("✅ Loaded Qwen in 4-bit (QLoRA mode).")

except Exception as e:
    print("⚠️ 4-bit failed, falling back to FP16 LoRA.")
    print("Reason:", str(e)[:200], "...")
    model_qwen = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        device_map=None,
    ).to(device)

✅ Loaded Qwen in 4-bit (QLoRA mode).


In [124]:
model_qwen.config.pad_token_id = tokenizer.pad_token_id
print("use_4bit =", use_4bit)

use_4bit = True


### Attach LoRA adapters:

In [125]:
if use_4bit:
    from peft import prepare_model_for_kbit_training
    model_qwen.gradient_checkpointing_enable()
    model_qwen = prepare_model_for_kbit_training(model_qwen)

target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=target_modules,
)

In [126]:
model_qwen = get_peft_model(model_qwen, lora_cfg)
model_qwen.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [127]:
model_qwen.config.use_cache = False

### Torch Dataset with prompt masking:

In [128]:
SYSTEM_MSG = "You are a calculator. Return ONLY the final result as digits (no words)."

def build_prompt_text(prompt: str) -> str:
    prompt = str(prompt).strip()
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": prompt},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [129]:
class QwenMathTorchDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=96):
        self.df = df.reset_index(drop=True)
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = row["prompt"]
        answer = str(row["answer"]).strip()

        prompt_text = build_prompt_text(prompt)
        full_text = prompt_text + answer + self.tok.eos_token

        enc_full = self.tok(full_text, truncation=True, max_length=self.max_len)
        enc_prompt = self.tok(prompt_text, truncation=True, max_length=self.max_len)

        input_ids = enc_full["input_ids"]
        attn = enc_full["attention_mask"]
        labels = input_ids.copy()

        # mask prompt tokens
        Lp = len(enc_prompt["input_ids"])
        labels[:Lp] = [-100] * Lp

        return {"input_ids": input_ids, "attention_mask": attn, "labels": labels}

In [130]:
def qwen_collator(features):
    batch = tokenizer.pad(features, padding=True, return_tensors="pt")

    batch["labels"][batch["attention_mask"] == 0] = -100
    return batch

### Create datasets:

In [131]:
TRAIN_N = 20_000   
VALID_N = 500      
MAX_LEN = 96       

train_ds = QwenMathTorchDataset(df_train.sample(TRAIN_N, random_state=SEED), tokenizer, max_len=MAX_LEN)
valid_ds = QwenMathTorchDataset(df_valid.sample(VALID_N, random_state=SEED), tokenizer, max_len=MAX_LEN)

In [132]:
len(train_ds)

20000

In [133]:
len(valid_ds)

500

### Training Arguments:

In [134]:
optim_name = "paged_adamw_8bit" if use_4bit else "adamw_torch"

args = TrainingArguments(
    output_dir="checkpoints_qwen_lora",
    overwrite_output_dir=True,
    do_train=True,
    do_eval=True,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    learning_rate=2e-4,
    num_train_epochs=1,
    warmup_steps=100,
    weight_decay=0.01,
    max_grad_norm=1.0,

    logging_strategy="steps",
    logging_steps=50,

    eval_strategy="epoch",      
    save_strategy="epoch",      
    save_total_limit=2,

    fp16=(device == "cuda"),
    remove_unused_columns=False,

    report_to=[],               
    optim=optim_name,

    dataloader_num_workers=0,   
)

### Train:

In [135]:
trainer = Trainer(
    model=model_qwen,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=qwen_collator,
)

In [136]:
trainer.train()

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss
1,0.010600,0.015418


TrainOutput(global_step=1250, training_loss=0.05351988229751587, metrics={'train_runtime': 8239.3038, 'train_samples_per_second': 2.427, 'train_steps_per_second': 0.152, 'total_flos': 1809553931727360.0, 'train_loss': 0.05351988229751587, 'epoch': 1.0})

It took around 2h 17m.

In [142]:
os.makedirs("checkpoints_qwen_lora/adapter", exist_ok=True)
trainer.save_model("checkpoints_qwen_lora/adapter")
tokenizer.save_pretrained("checkpoints_qwen_lora/adapter")
print("✅ Saved adapter to checkpoints_qwen_lora/adapter")

✅ Saved adapter to checkpoints_qwen_lora/adapter


### Generation and Evaluation:

In [143]:
@torch.no_grad()
def generate_answer_qwen(prompt, max_new_tokens=16):
    model_qwen.eval()
    prompt_text = build_prompt_text(prompt)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model_qwen.device)

    out = model_qwen.generate(
        **inputs,
        do_sample=False,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

    txt = tokenizer.decode(out[0], skip_special_tokens=True)
    nums = re.findall(r"\d+", txt)
    return nums[-1] if nums else ""

In [144]:
@torch.no_grad()
def exact_match_accuracy_qwen(df, max_items=2000, seed=0):
    df = df.sample(max_items, random_state=seed).reset_index(drop=True)

    correct = 0
    correct_add = correct_sub = 0
    total_add = total_sub = 0
    buckets, buckets_tot = {}, {}

    for _, r in df.iterrows():
        gt = str(r["answer"]).strip()
        pred = generate_answer_qwen(r["prompt"], max_new_tokens=16)
        ok = (pred == gt)
        correct += int(ok)

        if r["op"] == "+":
            total_add += 1
            correct_add += int(ok)
        else:
            total_sub += 1
            correct_sub += int(ok)

        k = int(r["len_a"])
        buckets_tot[k] = buckets_tot.get(k, 0) + 1
        buckets[k] = buckets.get(k, 0) + int(ok)

    overall = correct / len(df)
    add_acc = correct_add / max(total_add, 1)
    sub_acc = correct_sub / max(total_sub, 1)
    by_len_a = {k: buckets[k] / buckets_tot[k] for k in sorted(buckets_tot)}
    return overall, add_acc, sub_acc, by_len_a

In [147]:
def report_split_qwen(name, df, max_items=2000):
    overall, add_acc, sub_acc, by_len = exact_match_accuracy_qwen(df, max_items=max_items, seed=SEED)
    print(f"{name}: overall={overall:.4f} add={add_acc:.4f} sub={sub_acc:.4f} by_len_a={by_len}")

### evaluation:

In [149]:
report_split_qwen("valid_small", df_valid, max_items=100)

valid_small: overall=0.9500 add=0.9362 sub=0.9623 by_len_a={1: 1.0, 2: 0.96875, 3: 0.9666666666666667, 4: 0.85}


In [150]:
report_split_qwen("valid", df_valid, max_items=500)
report_split_qwen("test_seen", df_test_seen, max_items=1000)
report_split_qwen("test_unseen_numbers", df_test_unseen_numbers, max_items=1000)
report_split_qwen("test_unseen_formats", df_test_unseen_formats, max_items=1000)

valid: overall=0.9640 add=0.9577 sub=0.9708 by_len_a={1: 1.0, 2: 0.9923664122137404, 3: 0.959731543624161, 4: 0.917910447761194}
test_seen: overall=0.9750 add=0.9743 sub=0.9757 by_len_a={1: 1.0, 2: 0.9915966386554622, 3: 0.9756756756756757, 4: 0.948339483394834}
test_unseen_numbers: overall=0.9720 add=0.9710 sub=0.9729 by_len_a={1: 0.9844961240310077, 2: 0.9847908745247148, 3: 0.9710144927536232, 4: 0.9543726235741445}
test_unseen_formats: overall=0.9620 add=0.9671 sub=0.9571 by_len_a={1: 1.0, 2: 0.9844357976653697, 3: 0.945054945054945, 4: 0.9477611940298507}


## discussion:

- TinyTransformer (from scratch) reached near-perfect accuracy on test_seen and test_unseen_numbers once you deduplicated and fixed evaluation, but it was format brittle: test_unseen_formats ≈ 0.37. This is expected: a small char-level transformer trained on one canonical surface form tends to learn the mapping tightly coupled to formatting.

- Qwen2.5-0.5B + LoRA achieved strong accuracy across the board and, critically, excellent robustness to formatting: test_unseen_formats ≈ 0.96. This highlights the value of a pretrained instruction model: it already encodes broad textual patterns, so after a small task-specific fine-tune it generalizes to alternative whitespace/layout conventions.

However, Qwen’s supervised fine-tuning already took hours per epoch on your RTX 3050 Ti, and RL methods like PPO multiply compute due to sampling rollouts and extra forward passes (old policy, value head, KL/clipping). For a TP where you want to explore REINFORCE, actor-critic, and PPO, a smaller pretrained model is a better engineering choice.

That motivates proposing a third supervised experiment with a small GPT-2 family model (e.g., distilgpt2):

- It is much cheaper than Qwen, making PPO and repeated RL sweeps feasible on a single consumer GPU.

- It still benefits from pretraining (unlike the scratch TinyTransformer), so it should be less format-fragile than the scratch model after fine-tuning.

- Even if it does not match Qwen’s absolute performance, it is likely the best balance between capability and runtime for demonstrating RL fine-tuning algorithms in a controlled arithmetic setting.

## Experimentation (3) GPT-2 (distilgpt2) supervised fine-tune:

In [51]:
if device == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

In [52]:
# try:
#     from accelerate.state import AcceleratorState
#     AcceleratorState._reset_state()
# except Exception:
#     pass

In [53]:
import os
os.environ["ACCELERATE_USE_CPU"] = "false"
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"
os.environ["ACCELERATE_BYPASS_DEVICE_MAP"] = "true"

In [54]:
def normalize_prompt(p: str) -> str:
    p = str(p).strip()
    if p.endswith("="):
        return p + " "
    if p.endswith("= "):
        return p

    if "=" not in p:
        return p + " = "
    if p.endswith("="):
        return p + " "
    return p + " "

### GPT-2 tokenizer/model:

In [55]:
GPT2_NAME = "distilgpt2"
gpt2_tok = AutoTokenizer.from_pretrained(GPT2_NAME, use_fast=True)

GPT-2 has no pad token by default -> use eos as pad

In [56]:
if gpt2_tok.pad_token is None:
    gpt2_tok.pad_token = gpt2_tok.eos_token

In [57]:
gpt2 = AutoModelForCausalLM.from_pretrained(
    GPT2_NAME,
    torch_dtype=torch.float16 if device=="cuda" else torch.float32,
).to(device)

`torch_dtype` is deprecated! Use `dtype` instead!


In [58]:
gpt2.config.pad_token_id = gpt2_tok.pad_token_id
print("Loaded:", GPT2_NAME)

Loaded: distilgpt2


In [59]:
gpt2 = AutoModelForCausalLM.from_pretrained(GPT2_NAME).to(device)

### Torch Dataset with prompt masking:

In [60]:
class GPT2MathDataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_len=64):
        self.df = df.reset_index(drop=True)
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        prompt = normalize_prompt(r["prompt"])
        answer = str(r["answer"]).strip()

        prompt_text = prompt              # e.g. "12 + 7 = "
        full_text   = prompt_text + answer + self.tok.eos_token

        enc_full   = self.tok(full_text, truncation=True, max_length=self.max_len)
        enc_prompt = self.tok(prompt_text, truncation=True, max_length=self.max_len)

        input_ids = enc_full["input_ids"]
        attn      = enc_full["attention_mask"]
        labels    = input_ids.copy()

        Lp = len(enc_prompt["input_ids"])
        labels[:Lp] = [-100] * Lp   # mask prompt tokens

        return {"input_ids": input_ids, "attention_mask": attn, "labels": labels}

In [61]:
def gpt2_collator(features):
    # features: list of dicts with python lists
    max_len = max(len(f["input_ids"]) for f in features)

    input_ids = []
    attention_mask = []
    labels = []

    for f in features:
        ids = f["input_ids"]
        att = f["attention_mask"]
        lab = f["labels"]

        pad_len = max_len - len(ids)

        input_ids.append(ids + [gpt2_tok.pad_token_id] * pad_len)
        attention_mask.append(att + [0] * pad_len)
        labels.append(lab + [-100] * pad_len)  # pad labels with -100

    batch = {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }
    return batch

### Training Arguments:

In [62]:
TRAIN_N = 50_000     
VALID_N = 2_000
MAX_LEN = 64
EPOCHS  = 3

train_ds = GPT2MathDataset(df_train.sample(min(TRAIN_N, len(df_train)), random_state=SEED), gpt2_tok, max_len=MAX_LEN)
valid_ds = GPT2MathDataset(df_valid.sample(min(VALID_N, len(df_valid)), random_state=SEED), gpt2_tok, max_len=MAX_LEN)

print("train_ds:", len(train_ds), "valid_ds:", len(valid_ds))

train_ds: 50000 valid_ds: 2000


In [63]:
args = TrainingArguments(
    output_dir="checkpoints_gpt2_math",
    overwrite_output_dir=True,
    do_train=True,
    do_eval=True,

    per_device_train_batch_size=8 if device=="cuda" else 2,
    per_device_eval_batch_size=8 if device=="cuda" else 2,
    gradient_accumulation_steps=2,     
    

    learning_rate=5e-5,
    num_train_epochs=EPOCHS,
    warmup_steps=100,
    weight_decay=0.01,
    max_grad_norm=1.0,

    logging_strategy="steps",
    logging_steps=50,

    eval_strategy="epoch",             
    
    save_strategy="epoch",
    save_total_limit=2,

    fp16=(device=="cuda"),
    remove_unused_columns=False,

    report_to=[],                      
    
    dataloader_num_workers=0,          
    
)

### Train GPT-2:

In [64]:
use_bf16 = (device == "cuda") and torch.cuda.is_bf16_supported()
print("bf16 supported:", use_bf16)

bf16 supported: True


In [65]:
args = TrainingArguments(
    output_dir="checkpoints_gpt2_math",
    overwrite_output_dir=True,
    do_train=True,
    do_eval=True,

    per_device_train_batch_size=8 if device=="cuda" else 2,
    per_device_eval_batch_size=8 if device=="cuda" else 2,
    gradient_accumulation_steps=2,

    learning_rate=5e-5,
    num_train_epochs=EPOCHS,
    warmup_steps=100,
    weight_decay=0.01,

    logging_strategy="steps",
    logging_steps=50,

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    bf16=use_bf16,    
    fp16=False,       

    remove_unused_columns=False,
    report_to=[],
    dataloader_num_workers=0,
    max_grad_norm=1.0,
)

In [67]:
trainer = Trainer(
    model=gpt2,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=gpt2_collator,
)

In [68]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,1.649900,1.497823
2,1.348000,1.379214
3,1.314500,1.312830


TrainOutput(global_step=9375, training_loss=1.4995208601888022, metrics={'train_runtime': 975.8851, 'train_samples_per_second': 153.707, 'train_steps_per_second': 9.607, 'total_flos': 329348223074304.0, 'train_loss': 1.4995208601888022, 'epoch': 3.0})

In [70]:
# Save best/last style: Trainer saves per epoch; also save a final snapshot
os.makedirs("checkpoints_gpt2_math/final", exist_ok=True)
trainer.save_model("checkpoints_gpt2_math/final")
gpt2_tok.save_pretrained("checkpoints_gpt2_math/final")
print("✅ Saved GPT2 model to checkpoints_gpt2_math/final")

✅ Saved GPT2 model to checkpoints_gpt2_math/final


### GPT-2 generation and evaluation:

In [71]:
@torch.no_grad()
def generate_answer_gpt2(prompt, max_new_tokens=10):
    gpt2.eval()
    p = normalize_prompt(prompt)

    inputs = gpt2_tok(p, return_tensors="pt").to(device)
    out = gpt2.generate(
        **inputs,
        do_sample=False,
        max_new_tokens=max_new_tokens,
        eos_token_id=gpt2_tok.eos_token_id,
        pad_token_id=gpt2_tok.pad_token_id,
    )
    txt = gpt2_tok.decode(out[0], skip_special_tokens=True)

    # Extract last integer from generated text
    nums = re.findall(r"\d+", txt)
    return nums[-1] if nums else ""

@torch.no_grad()
def exact_match_accuracy_gpt2(df, max_items=2000, seed=0):
    df = df.sample(min(max_items, len(df)), random_state=seed).reset_index(drop=True)

    correct = 0
    correct_add = correct_sub = 0
    total_add = total_sub = 0
    buckets, buckets_tot = {}, {}

    for _, r in df.iterrows():
        gt = str(r["answer"]).strip()
        pred = generate_answer_gpt2(r["prompt"], max_new_tokens=10)
        ok = (pred == gt)
        correct += int(ok)

        if r["op"] == "+":
            total_add += 1
            correct_add += int(ok)
        else:
            total_sub += 1
            correct_sub += int(ok)

        k = int(r["len_a"])
        buckets_tot[k] = buckets_tot.get(k, 0) + 1
        buckets[k] = buckets.get(k, 0) + int(ok)

    overall = correct / len(df)
    add_acc = correct_add / max(total_add, 1)
    sub_acc = correct_sub / max(total_sub, 1)
    by_len_a = {k: buckets[k] / buckets_tot[k] for k in sorted(buckets_tot)}
    return overall, add_acc, sub_acc, by_len_a

def report_split_gpt2(name, df, max_items=2000):
    overall, add_acc, sub_acc, by_len = exact_match_accuracy_gpt2(df, max_items=max_items, seed=SEED)
    print(f"{name}: overall={overall:.4f} add={add_acc:.4f} sub={sub_acc:.4f} by_len_a={by_len}")

In [72]:
report_split_gpt2("valid", df_valid, max_items=2000)
report_split_gpt2("test_seen", df_test_seen, max_items=5000)
report_split_gpt2("test_unseen_numbers", df_test_unseen_numbers, max_items=5000)
report_split_gpt2("test_unseen_formats", df_test_unseen_formats, max_items=5000)

valid: overall=0.0035 add=0.0050 sub=0.0020 by_len_a={1: 0.01968503937007874, 2: 0.0, 3: 0.0, 4: 0.0036231884057971015}
test_seen: overall=0.0040 add=0.0032 sub=0.0048 by_len_a={1: 0.030852994555353903, 2: 0.002364066193853428, 3: 0.0, 4: 0.0}
test_unseen_numbers: overall=0.0426 add=0.0699 sub=0.0139 by_len_a={1: 0.34471544715447155, 2: 0.0, 3: 0.0005998800239952009, 4: 0.0}
test_unseen_formats: overall=0.0068 add=0.0082 sub=0.0053 by_len_a={1: 0.047933884297520664, 2: 0.003930817610062893, 3: 0.0, 4: 0.0}


### We can afford further training:

In [78]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer

GPT2_NAME = "distilgpt2"
last_ckpt = last_ckpt  # keep your checkpoint path

gpt2_tok = AutoTokenizer.from_pretrained(GPT2_NAME, use_fast=True)
if gpt2_tok.pad_token is None:
    gpt2_tok.pad_token = gpt2_tok.eos_token

gpt2 = AutoModelForCausalLM.from_pretrained(last_ckpt).to(device)
gpt2.config.pad_token_id = gpt2_tok.pad_token_id


In [79]:
NEW_TOTAL_EPOCHS = 15
args.num_train_epochs = NEW_TOTAL_EPOCHS

In [80]:
trainer = Trainer(
    model=gpt2,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=gpt2_collator,
)

trainer.train()   

Epoch,Training Loss,Validation Loss
1,1.166800,1.077072
2,0.844000,0.900449
3,0.766100,0.810460
4,0.662800,0.727113
5,0.536200,0.660567
6,0.476600,0.584341
7,0.433500,0.535722
8,0.284800,0.492390
9,0.241300,0.437035
10,0.170700,0.396908


TrainOutput(global_step=46875, training_loss=0.4256875882364909, metrics={'train_runtime': 4853.0309, 'train_samples_per_second': 154.543, 'train_steps_per_second': 9.659, 'total_flos': 1646606384234496.0, 'train_loss': 0.4256875882364909, 'epoch': 15.0})

In [81]:
save_dir = "checkpoints_gpt2_math/final_after_resume"
trainer.save_model(save_dir)
gpt2_tok.save_pretrained(save_dir)
print("✅ Saved to:", save_dir)

✅ Saved to: checkpoints_gpt2_math/final_after_resume


In [85]:
report_split_gpt2("valid", df_valid, max_items=2000)
report_split_gpt2("test_seen", df_test_seen, max_items=5000)
report_split_gpt2("test_unseen_numbers", df_test_unseen_numbers, max_items=5000)
report_split_gpt2("test_unseen_formats", df_test_unseen_formats, max_items=5000)

valid: overall=0.0270 add=0.0455 sub=0.0081 by_len_a={1: 0.08267716535433071, 2: 0.05263157894736842, 3: 0.01, 4: 0.0}
test_seen: overall=0.0238 add=0.0394 sub=0.0084 by_len_a={1: 0.0544464609800363, 2: 0.06304176516942474, 3: 0.004514672686230248, 4: 0.0007102272727272727}
test_unseen_numbers: overall=0.0380 add=0.0582 sub=0.0168 by_len_a={1: 0.2910569105691057, 2: 0.002223869532987398, 3: 0.004799040191961607, 4: 0.0}
test_unseen_formats: overall=0.0324 add=0.0540 sub=0.0098 by_len_a={1: 0.12396694214876033, 2: 0.05817610062893082, 3: 0.006347374495095211, 4: 0.0014388489208633094}


## Descussion: which model is worth using for RL?


### 1) Tiny char-Transformer:

- Strength: It already reaches near-perfect exact-match accuracy on the canonical format (“a op b =”)

- Practicality: It is fast to sample from and cheap to update, which is the core requirement for policy-gradient methods. REINFORCE/PPO need many rollouts. the bottleneck is generation and backprop, not dataset size.

- RL suitability: Excellent. The action space is tiny (character tokens) and you can constrain decoding to digits+EOS cleanly. That makes reward signals stable, reduces variance, and makes REINFORCE work reliably.

### 2) Qwen2.5-0.5B-Instruct + QLoRA:

- Strength: It generalizes well and gives high accuracy even on “unseen formats” in our results, which is impressive.

- Practicality: For RL it’s a poor fit on an RTX 3050 Ti because RL requires tens/hundreds of thousands of sampled sequences. our supervised run already took ~hours per epoch. so RL would multiply that cost because you must sample and update repeatedly (and PPO is heavier than supervised).

- RL suitability: Possible in theory, but not efficient for your hardware/time budget. we’d end up either:

    - running too few RL steps to show clear improvement.
    
    - shrinking the rollout budget so much that variance dominates and results become noisy/unreliable.

## 3) GPT-2 / small GPT-like models:

- Outcome: Despite decreasing losses, exact-match accuracy stayed ~0 a, meaning it did not learn to reliably generate correct arithmetic answers in our setup. Loss going down here mainly reflects token-level fitting, not algorithmic correctness.

- RL suitability: Not worth it. RL fine-tuning needs a base policy that already produces reasonable outputs. otherwise rewards are almost always zero, gradients vanish into noise, and REINFORCE becomes unstable.

## Conclusion:

- We will Start with REINFORCE and moving-average baseline.

- Then we will upgrade to value baseline and PPO if time allows.


# Reinforcement Learning Fine-Tuning:

### RL setup:

In [93]:
import os, re, math, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

In [94]:
torch.manual_seed(0)
random.seed(0)
np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [160]:
import torch
import os, random, numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [162]:
SUP_CKPT = "checkpoints/sup_scratch_best_by_em.pt"  

model_rl = TinyTransformer(
    vocab_size=vocab_size,
    d_model=256,   
    n_layers=4,    
    heads=4,       
    max_len=128    
).to(device)

state = torch.load(SUP_CKPT, map_location=device, weights_only=True)

model_rl.load_state_dict(state, strict=True)
model_rl.eval()

print("Loaded:", SUP_CKPT)
print("Params (M):", sum(p.numel() for p in model_rl.parameters())/1e6)

Loaded: checkpoints/sup_scratch_best_by_em.pt
Params (M): 3.168273


### Robust parsing + reward:

In [164]:
def normalize_prompt(prompt: str) -> str:
    p = str(prompt).strip()
    # ensure canonical form "... ="
    if p.endswith("="):
        p = p + " "
    elif not p.endswith("= "):
        if "=" not in p:
            p = p + " = "
        else:
            if p.endswith("="):
                p += " "
            elif not p.endswith("= "):
                p = p.rstrip() + " "
                if not p.endswith("= "):
                    # best effort
                    p = p.replace("=", "= ").rstrip() + " "
    return p

In [166]:
_prompt_re = re.compile(r"^\s*(\d+)\s*([+\-])\s*(\d+)\s*=\s*$")

In [168]:
def compute_answer_from_prompt(prompt: str) -> str:
    p = normalize_prompt(prompt)
    m = _prompt_re.match(p)
    if not m:
        m2 = re.search(r"(\d+)\s*([+\-])\s*(\d+)", p)
        if not m2:
            raise ValueError(f"Cannot parse prompt: {prompt}")
        a, op, b = int(m2.group(1)), m2.group(2), int(m2.group(3))
    else:
        a, op, b = int(m.group(1)), m.group(2), int(m.group(3))

    if op == "+":
        return str(a + b)
    else:
        return str(a - b)

In [169]:
def get_gt_answer(row) -> str:
    if "answer" in row and pd.notna(row["answer"]):
        return str(row["answer"]).strip()
    return compute_answer_from_prompt(row["prompt"])

In [170]:
def exact_match_reward(pred: str, gt: str) -> float:
    return 1.0 if str(pred).strip() == str(gt).strip() else 0.0

### Sampling rollout (policy) with logprobs:

In [171]:
DIGIT_IDS = [vocab2id[str(d)] for d in range(10)]
ALLOWED_IDS = torch.tensor(DIGIT_IDS + [EOS_ID], dtype=torch.long, device=device)

In [173]:
@torch.no_grad()
def greedy_answer(model, prompt, max_answer_len=8) -> str:
    model.eval()
    p = normalize_prompt(prompt)
    ids = encode(p)
    input_ids = torch.tensor(ids[:-1], dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_answer_len):
        logits = model(input_ids)[0, -1, :]  # (V,)
        masked = torch.full_like(logits, float("-inf"))
        masked[ALLOWED_IDS] = logits[ALLOWED_IDS]
        next_id = int(torch.argmax(masked).item())
        if next_id == EOS_ID:
            break
        input_ids = torch.cat([input_ids, torch.tensor([[next_id]], device=device)], dim=1)

    gen_ids = input_ids[0].tolist()
    gen_text = decode(gen_ids[len(ids[:-1]):]).strip()
    return gen_text

In [174]:
def sample_rollout(model, prompt, max_answer_len=8, temperature=1.0):
    """
    Returns:
      pred_text (str)
      logprob_sum (torch scalar)
      entropy_sum (torch scalar)
      tokens (list[int])
    """
    model.eval()  # sampling is fine in eval mode; gradients handled separately elsewhere
    p = normalize_prompt(prompt)

    ids = encode(p)
    input_ids = torch.tensor(ids[:-1], dtype=torch.long, device=device).unsqueeze(0)

    logps = []
    ents = []
    gen_tokens = []

    for _ in range(max_answer_len):
        logits = model(input_ids)[0, -1, :]  

        masked = torch.full_like(logits, float("-inf"))
        masked[ALLOWED_IDS] = logits[ALLOWED_IDS]

        masked = masked / max(temperature, 1e-6)
        probs = torch.softmax(masked, dim=-1)

        dist = torch.distributions.Categorical(probs=probs)
        next_id = dist.sample()  
        logp = dist.log_prob(next_id)
        ent = dist.entropy()

        next_id_int = int(next_id.item())
        if next_id_int == EOS_ID:
            logps.append(logp)
            ents.append(ent)
            break

        logps.append(logp)
        ents.append(ent)
        gen_tokens.append(next_id_int)

        input_ids = torch.cat([input_ids, next_id.view(1,1)], dim=1)

    pred_text = decode(gen_tokens).strip()
    logprob_sum = torch.stack(logps).sum() if len(logps) else torch.tensor(0.0, device=device)
    entropy_sum = torch.stack(ents).sum() if len(ents) else torch.tensor(0.0, device=device)
    return pred_text, logprob_sum, entropy_sum, gen_tokens

### REINFORCE with moving-average baseline:

In [175]:
assert "prompt" in df_rl.columns, "df_rl must contain a 'prompt' column."

if "answer" not in df_rl.columns:
    df_rl = df_rl.copy()
    df_rl["answer"] = df_rl["prompt"].apply(compute_answer_from_prompt)

df_rl = df_rl.reset_index(drop=True)
print("RL rows:", len(df_rl))
df_rl.head()

RL rows: 50000


,prompt,answer,op,a,b,fmt_id,len_a,len_b,len_ans
0,8 - 4 =,4,-,8,4,0,1,1,1
1,851 + 53 =,904,+,851,53,0,3,2,3
2,56 + 50 =,106,+,56,50,0,2,2,3
3,6525 - 5 =,6520,-,6525,5,0,4,1,4
4,4699 - 5 =,4694,-,4699,5,0,4,1,4


### REINFORCE training loop:

In [176]:
def evaluate_policy_exact_match(model, df, max_items=2000, seed=0):
    df_s = df.sample(min(max_items, len(df)), random_state=seed).reset_index(drop=True)

    correct = 0
    correct_add = 0
    correct_sub = 0
    total_add = 0
    total_sub = 0
    buckets, buckets_tot = {}, {}

    for _, r in df_s.iterrows():
        p = r["prompt"]
        gt = get_gt_answer(r)
        pred = greedy_answer(model, p, max_answer_len=8)
        ok = (pred == gt)

        correct += int(ok)
        op = r.get("op", None)
        if op is None:
            op = "+" if "+" in str(p) else "-"

        if op == "+":
            total_add += 1
            correct_add += int(ok)
        else:
            total_sub += 1
            correct_sub += int(ok)

        if "len_a" in r:
            k = int(r["len_a"])
        else:
            m = re.search(r"(\d+)\s*([+\-])\s*(\d+)", str(p))
            k = len(m.group(1)) if m else 0

        buckets_tot[k] = buckets_tot.get(k, 0) + 1
        buckets[k] = buckets.get(k, 0) + int(ok)

    overall = correct / len(df_s)
    add_acc = correct_add / max(total_add, 1)
    sub_acc = correct_sub / max(total_sub, 1)
    by_len_a = {k: buckets[k] / buckets_tot[k] for k in sorted(buckets_tot)}
    return overall, add_acc, sub_acc, by_len_a

In [177]:
def train_reinforce_baseline(
    model,
    df_rl,
    steps=2000,
    batch_size=64,
    lr=5e-6,
    max_answer_len=8,
    temperature=1.0,
    baseline_beta=0.95,
    entropy_coef=0.01,
    grad_clip=1.0,
    eval_every=200,
    seed=0,
):
    random.seed(seed)
    torch.manual_seed(seed)

    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    # moving average baseline
    b = 0.0

    history = []

    for step in range(1, steps + 1):
        # sample batch
        idxs = np.random.randint(0, len(df_rl), size=batch_size)
        batch = df_rl.iloc[idxs]

        logprob_sums = []
        entropy_sums = []
        rewards = []

        # rollout per sample (simplest + robust)
        for _, r in batch.iterrows():
            p = r["prompt"]
            gt = get_gt_answer(r)

            pred, logp_sum, ent_sum, _ = sample_rollout(
                model, p,
                max_answer_len=max_answer_len,
                temperature=temperature
            )
            rew = exact_match_reward(pred, gt)

            logprob_sums.append(logp_sum)
            entropy_sums.append(ent_sum)
            rewards.append(rew)

        rewards_t = torch.tensor(rewards, dtype=torch.float32, device=device)
        logprob_sums_t = torch.stack(logprob_sums) if len(logprob_sums) else torch.zeros(batch_size, device=device)
        entropy_sums_t = torch.stack(entropy_sums) if len(entropy_sums) else torch.zeros(batch_size, device=device)

        # update baseline
        r_mean = float(rewards_t.mean().item())
        b = baseline_beta * b + (1 - baseline_beta) * r_mean

        # advantage
        adv = rewards_t - b
        # (optional) normalize advantage for stability
        if adv.std() > 1e-6:
            adv = (adv - adv.mean()) / (adv.std() + 1e-6)

        # maximize: E[adv * logpi + entropy_coef * entropy]
        # minimize negative:
        loss = -(adv.detach() * logprob_sums_t).mean() - entropy_coef * entropy_sums_t.mean()

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        if step % 20 == 0 or step == 1:
            history.append({
                "step": step,
                "batch_reward_mean": r_mean,
                "baseline": b,
                "loss": float(loss.item()),
            })

        if step % eval_every == 0:
            model.eval()
            v_overall, v_add, v_sub, v_bylen = evaluate_policy_exact_match(model, df_valid, max_items=2000, seed=seed)
            print(f"[step {step:5d}] batch_reward={r_mean:.3f} baseline={b:.3f} loss={loss.item():.4f}")
            print(f"          VALID EM: overall={v_overall:.3f} add={v_add:.3f} sub={v_sub:.3f} by_len_a={v_bylen}")
            model.train()

    return history

### Baseline evaluation (before applying RL):

In [178]:
model_rl.eval()
overall, add_acc, sub_acc, by_len = evaluate_policy_exact_match(model_rl, df_valid, max_items=2000, seed=0)

print(f"BEFORE RL - valid: overall={overall:.4f} add={add_acc:.4f} sub={sub_acc:.4f} by_len_a={by_len}")

BEFORE RL - valid: overall=0.9960 add=0.9980 sub=0.9939 by_len_a={1: 1.0, 2: 1.0, 3: 0.9957142857142857, 4: 0.9909420289855072}


### Run REINFORCE baseline:

In [180]:
history = train_reinforce_baseline(
    model=model_rl,
    df_rl=df_rl,
    steps=300,
    batch_size=64,
    lr=5e-6,
    max_answer_len=8,
    temperature=1.0,
    baseline_beta=0.95,
    entropy_coef=0.01,
    eval_every=50,
    seed=0,
)

# save RL-tuned weights
os.makedirs("checkpoints_rl", exist_ok=True)
torch.save(model_rl.state_dict(), "checkpoints_rl/reinforce_baseline_last.pt")
print("Saved: checkpoints_rl/reinforce_baseline_last.pt")

[step    50] batch_reward=1.000 baseline=0.919 loss=0.0000
          VALID EM: overall=0.996 add=0.998 sub=0.994 by_len_a={1: 1.0, 2: 1.0, 3: 0.9957142857142857, 4: 0.9909420289855072}
[step   100] batch_reward=1.000 baseline=0.988 loss=-0.0000
          VALID EM: overall=0.996 add=0.998 sub=0.994 by_len_a={1: 1.0, 2: 1.0, 3: 0.9957142857142857, 4: 0.9909420289855072}
[step   150] batch_reward=0.984 baseline=0.993 loss=-0.1054
          VALID EM: overall=0.996 add=0.998 sub=0.994 by_len_a={1: 1.0, 2: 1.0, 3: 0.9957142857142857, 4: 0.9909420289855072}
[step   200] batch_reward=1.000 baseline=0.993 loss=-0.0000
          VALID EM: overall=0.997 add=0.999 sub=0.994 by_len_a={1: 1.0, 2: 1.0, 3: 0.9957142857142857, 4: 0.9927536231884058}
[step   250] batch_reward=1.000 baseline=0.995 loss=-0.0005
          VALID EM: overall=0.997 add=0.999 sub=0.995 by_len_a={1: 1.0, 2: 1.0, 3: 0.9957142857142857, 4: 0.9945652173913043}
[step   300] batch_reward=1.000 baseline=0.996 loss=-0.0000
          V

### Evaluate after RL:

In [181]:
model_rl.eval()

for name, df_ in [
    ("valid", df_valid),
    ("test_seen", df_test_seen if "df_test_seen" in globals() else df_valid),
    ("test_unseen_numbers", df_test_unseen_numbers if "df_test_unseen_numbers" in globals() else df_valid),
    ("test_unseen_formats", df_test_unseen_formats if "df_test_unseen_formats" in globals() else df_valid),
]:
    overall, add_acc, sub_acc, by_len = evaluate_policy_exact_match(model_rl, df_, max_items=5000, seed=0)
    print(f"AFTER RL - {name}: overall={overall:.4f} add={add_acc:.4f} sub={sub_acc:.4f} by_len_a={by_len}")

AFTER RL - valid: overall=0.9972 add=0.9984 sub=0.9959 by_len_a={1: 1.0, 2: 1.0, 3: 0.9948006932409013, 4: 0.9962935507783544}
AFTER RL - test_seen: overall=0.9962 add=0.9980 sub=0.9944 by_len_a={1: 1.0, 2: 1.0, 3: 0.9971783295711061, 4: 0.9900568181818182}
AFTER RL - test_unseen_numbers: overall=0.9932 add=0.9965 sub=0.9898 by_len_a={1: 1.0, 2: 0.9903632320237212, 3: 0.9934013197360528, 4: 0.9926953981008035}
AFTER RL - test_unseen_formats: overall=0.3748 add=0.3606 sub=0.3897 by_len_a={1: 0.46611570247933887, 2: 0.3356918238993711, 3: 0.3866128101557992, 4: 0.35611510791366907}


Quick sanity check:

In [182]:
tests = ["12 + 7 =", "45 - 19 =", "103 + 88 =", "345 + 678 ="]
for p in tests:
    print(p, "->", greedy_answer(model_rl, p, max_answer_len=8))

12 + 7 = -> 19
45 - 19 = -> 26
103 + 88 = -> 191
345 + 678 = -> 1023


## Discussion:

We propose using PPO to target prompt-format generalization because the supervised model already achieves near-ceiling performance on canonical prompts, leaving little room for improvement under REINFORCE with sparse rewards. PPO enables stable policy updates through its clipped objective, allowing us to optimize for robustness to formatting variations without degrading arithmetic accuracy. By training PPO on an RL dataset composed of format-augmented prompts (e.g., missing spaces, different delimiters, or instruction-style prefixes) and using a reward function centered on numeric correctness with auxiliary penalties for invalid outputs, PPO explicitly pressures the model to become invariant to surface form changes. We therefore expect PPO to preserve performance on seen and unseen-number splits while substantially improving accuracy on the unseen-format split.

# PPO on format-augmented prompts:

In [185]:
def infer_tiny_cfg_from_state_dict(state):
    d_model = state["embed.weight"].shape[1]
    max_len = state["pos.pe"].shape[0]
    block_ids = []
    for k in state.keys():
        m = re.match(r"blocks\.(\d+)\.", k)
        if m:
            block_ids.append(int(m.group(1)))
    n_layers = (max(block_ids) + 1) if block_ids else 0

    for h in [8, 6, 4, 2, 1]:
        if d_model % h == 0:
            heads = h
            break
    return dict(d_model=d_model, max_len=max_len, n_layers=n_layers, heads=heads)

In [186]:
def load_tiny_from_ckpt(ckpt_path, vocab_size):
    state = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    cfg = infer_tiny_cfg_from_state_dict(state)
    model = TinyTransformer(vocab_size=vocab_size, **cfg).to(device)
    model.load_state_dict(state, strict=True)
    model.eval()
    print("Loaded ckpt:", ckpt_path)
    print("Inferred cfg:", cfg)
    return model, cfg

In [187]:
SUP_CKPT = "checkpoints/sup_scratch_best_by_em.pt"  # change if needed
model_base, base_cfg = load_tiny_from_ckpt(SUP_CKPT, vocab_size=vocab_size)

Loaded ckpt: checkpoints/sup_scratch_best_by_em.pt
Inferred cfg: {'d_model': 256, 'max_len': 128, 'n_layers': 4, 'heads': 8}


### Build RL prompts that change the format:

In [188]:
def parse_prompt_canonical(prompt: str):
    s = prompt.strip()
    s = s.replace("=", " = ")
    s = " ".join(s.split())  
    parts = s.split()
    a = int(parts[0]); op = parts[1]; b = int(parts[2])
    return a, op, b

In [189]:
def format_variants(a, op, b):
    # only digits, space, +, -, =
    variants = [
        f"{a}{op}{b}=",
        f"{a} {op}{b}=",
        f"{a}{op} {b}=",
        f"{a} {op} {b}=",
        f"{a} {op} {b} =",
        f"{a}  {op}  {b}  =",
        f" {a} {op} {b} =",
        f"{a} {op} {b}= ",
        f"{a} {op} {b}  = ",
        f"{a} {op}  {b} =",
    ]
    return variants

In [190]:
def make_df_rl_formats(df_src, n_samples=50000, seed=0):
    rng = random.Random(seed)
    rows = []
    for _ in range(n_samples):
        r = df_src.iloc[rng.randrange(len(df_src))]
        a, op, b = parse_prompt_canonical(str(r["prompt"]))
        ans = str(r["answer"]).strip()
        prompt_var = rng.choice(format_variants(a, op, b))
        rows.append((prompt_var, ans))
    return pd.DataFrame(rows, columns=["prompt", "answer"])

In [192]:
df_rl_formats = make_df_rl_formats(df_rl, n_samples=min(50000, len(df_rl)*5), seed=0)

In [193]:
df_rl_formats.head()

,prompt,answer
0,174 + 680 =,854
1,9130 - 765 =,8365
2,1 + 843=,844
3,92 - 2 =,90
4,230 - 12 =,218


In [194]:
len(df_rl_formats)

50000

### PPO model wrapper: add a value head

In [195]:
class TinyWithValue(nn.Module):
    def __init__(self, base: TinyTransformer):
        super().__init__()
        self.base = base
        d_model = base.embed.embedding_dim
        self.v_head = nn.Linear(d_model, 1)

    def forward_logits_and_value(self, x):
        B, T = x.size()
        if T > self.base.max_len:
            raise ValueError(f"T={T} > max_len={self.base.max_len}")
        mask = torch.tril(torch.ones(T, T, device=x.device))
        h = self.base.embed(x)
        h = self.base.pos(h)
        for block in self.base.blocks:
            h = block(h, mask)
        h = self.base.ln(h)
        logits = self.base.fc(h)  
        v = self.v_head(h[:, -1, :]).squeeze(-1)  
        return logits, v

In [196]:
model_rl = TinyWithValue(model_base).to(device)

### PPO rollout and update:

In [197]:
DIGIT_IDS = [vocab2id[str(d)] for d in range(10)]
ALLOWED_IDS = set(DIGIT_IDS + [EOS_ID])

@torch.no_grad()
def sample_answer_and_logp(model_v: TinyWithValue, prompt: str, max_answer_len=8, temperature=1.0):
    s = str(prompt)
    if s.endswith("="):
        s = s + " "
    ids = encode(s)
    ctx = torch.tensor(ids[:-1], dtype=torch.long, device=device).unsqueeze(0)  # (1,T)

    _, v = model_v.forward_logits_and_value(ctx)

    action_ids = []
    action_logps = []
    action_entropies = []

    for _ in range(max_answer_len):
        logits, _ = model_v.forward_logits_and_value(ctx)
        next_logits = logits[0, -1, :] / max(temperature, 1e-8)

        masked = torch.full_like(next_logits, float("-inf"))
        for i in ALLOWED_IDS:
            masked[i] = next_logits[i]

        probs = torch.softmax(masked, dim=-1)
        dist = torch.distributions.Categorical(probs=probs)
        next_id = int(dist.sample().item())
        logp = dist.log_prob(torch.tensor(next_id, device=device))
        ent = dist.entropy()

        if next_id == EOS_ID:
            break

        action_ids.append(next_id)
        action_logps.append(logp)
        action_entropies.append(ent)

        ctx = torch.cat([ctx, torch.tensor([[next_id]], device=device)], dim=1)

    gen = decode(action_ids).strip()
    if len(action_logps) == 0:
        sum_logp = torch.tensor(0.0, device=device)
        mean_ent = torch.tensor(0.0, device=device)
    else:
        sum_logp = torch.stack(action_logps).sum()
        mean_ent = torch.stack(action_entropies).mean()

    return {
        "prompt": s,
        "prompt_ctx_ids": ids[:-1],
        "action_ids": action_ids,
        "gen": gen,
        "old_logp": sum_logp.detach(),
        "old_value": v.detach().squeeze(0),
        "entropy": mean_ent.detach(),
    }

In [198]:
def reward_exact(gen: str, gold: str):
    return 1.0 if str(gen).strip() == str(gold).strip() else 0.0

In [199]:
def build_batch_rollouts(model_v, df_batch, max_answer_len=8, temperature=1.0):
    rollouts = []
    for _, r in df_batch.iterrows():
        out = sample_answer_and_logp(model_v, r["prompt"], max_answer_len=max_answer_len, temperature=temperature)
        out["gold"] = str(r["answer"]).strip()
        out["reward"] = reward_exact(out["gen"], out["gold"])
        rollouts.append(out)
    return rollouts

In [200]:
def compute_logp_for_actions(model_v, prompt_ctx_ids, action_ids):
    if len(action_ids) == 0:
        return torch.tensor(0.0, device=device), torch.tensor(0.0, device=device)

    seq = torch.tensor(prompt_ctx_ids + action_ids, dtype=torch.long, device=device).unsqueeze(0)  # (1,T)
    logits, v = model_v.forward_logits_and_value(seq[:, :-1])  # predict next tokens for positions
    
    start = len(prompt_ctx_ids) - 1
    logps = []
    for i, a_id in enumerate(action_ids):
        t = start + i
        step_logits = logits[0, t, :]
        masked = torch.full_like(step_logits, float("-inf"))
        for j in ALLOWED_IDS:
            masked[j] = step_logits[j]
        logprob = torch.log_softmax(masked, dim=-1)[a_id]
        logps.append(logprob)
    return torch.stack(logps).sum(), v.squeeze(0)

In [201]:
def ppo_update(model_v, rollouts, optimizer, clip_eps=0.2, vf_coef=0.5, ent_coef=0.01, ppo_epochs=2):
    rewards = torch.tensor([ro["reward"] for ro in rollouts], device=device, dtype=torch.float32)
    old_logp = torch.stack([ro["old_logp"] for ro in rollouts]).to(device)
    old_v    = torch.stack([ro["old_value"] for ro in rollouts]).to(device)

    adv = (rewards - old_v).detach()
    if adv.numel() > 1:
        adv = (adv - adv.mean()) / (adv.std(unbiased=False) + 1e-8)

    losses = []
    for _ in range(ppo_epochs):
        new_logps = []
        new_vs = []
        ents = []
        for ro in rollouts:
            lp, v = compute_logp_for_actions(model_v, ro["prompt_ctx_ids"], ro["action_ids"])
            new_logps.append(lp)
            new_vs.append(v)
            ents.append(ro["entropy"].to(device))

        new_logp = torch.stack(new_logps)
        new_v = torch.stack(new_vs)
        entropy = torch.stack(ents).mean()

        ratio = torch.exp(new_logp - old_logp)  
        unclipped = ratio * adv
        clipped = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * adv
        policy_loss = -torch.mean(torch.min(unclipped, clipped))

        value_loss = torch.mean((new_v - rewards) ** 2)
        loss = policy_loss + vf_coef * value_loss - ent_coef * entropy

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_v.parameters(), 1.0)
        optimizer.step()

        losses.append(loss.item())

    return {
        "loss": float(sum(losses)/len(losses)),
        "policy_loss": float(policy_loss.item()),
        "value_loss": float(value_loss.item()),
        "entropy": float(entropy.item()),
        "batch_reward": float(rewards.mean().item()),
    }

In [204]:
@torch.no_grad()
def generate_answer_scratch(model, prompt: str, max_answer_len=8, temperature=1.0):
    """
    model: TinyTransformer (not TinyWithValue)
    prompt: string like "12 + 7 =" with any spacing variants
    returns: decoded answer string (digits) without EOS
    """
    model.eval()

    s = str(prompt)
    if s.endswith("="):
        s = s + " "

    ids = encode(s)  
    ctx = torch.tensor(ids[:-1], dtype=torch.long, device=device).unsqueeze(0)

    out_ids = []
    for _ in range(max_answer_len):
        logits = model(ctx)                 
        next_logits = logits[0, -1, :]     

        masked = torch.full_like(next_logits, float("-inf"))
        for i in ALLOWED_IDS:
            masked[i] = next_logits[i]

        masked = masked / max(temperature, 1e-8)
        probs = torch.softmax(masked, dim=-1)

        next_id = int(torch.multinomial(probs, num_samples=1).item())
        if next_id == EOS_ID:
            break

        out_ids.append(next_id)
        ctx = torch.cat([ctx, torch.tensor([[next_id]], device=device)], dim=1)

    return decode(out_ids).strip()

In [205]:
def _extract_a_len_from_prompt(prompt: str):
    m = re.search(r"\d+", str(prompt))
    if not m:
        return 0
    return len(m.group(0))

In [206]:
def report_split(name, df, max_items=2000, max_answer_len=8, temperature=1.0):
    """
    Prints:
      overall EM, add EM, sub EM, and by_len_a dict.
    Uses global model_rl (TinyWithValue) OR model_base if you prefer.
    """
    # choose the policy model you are training with PPO
    # model_rl is TinyWithValue -> use model_rl.base for generation
    policy = model_rl.base if hasattr(model_rl, "base") else model_rl

    sample = df if (max_items is None or len(df) <= max_items) else df.sample(max_items, random_state=0)

    total = 0
    ok = 0
    add_total = add_ok = 0
    sub_total = sub_ok = 0
    by_len_ok = {}
    by_len_total = {}

    for _, r in sample.iterrows():
        p = str(r["prompt"])
        gt = str(r["answer"]).strip()
        pred = generate_answer_scratch(policy, p, max_answer_len=max_answer_len, temperature=temperature)

        em = (pred.strip() == gt)
        total += 1
        ok += int(em)

        op = "+" if "+" in p else ("-" if "-" in p else None)
        la = _extract_a_len_from_prompt(p)

        by_len_total[la] = by_len_total.get(la, 0) + 1
        by_len_ok[la] = by_len_ok.get(la, 0) + int(em)

        if op == "+":
            add_total += 1
            add_ok += int(em)
        elif op == "-":
            sub_total += 1
            sub_ok += int(em)

    overall = ok / max(total, 1)
    add_acc = add_ok / max(add_total, 1)
    sub_acc = sub_ok / max(sub_total, 1)

    by_len = {k: by_len_ok.get(k, 0) / by_len_total[k] for k in sorted(by_len_total.keys())}

    print(f"{name}: overall={overall:.4f} add={add_acc:.4f} sub={sub_acc:.4f} by_len_a={by_len}")
    return overall, add_acc, sub_acc, by_len

### PPO training loop:

In [207]:
def train_ppo_formats(
    model_v,
    df_rl_formats,
    steps=1000,
    batch_size=64,
    lr=3e-6,
    temperature=1.0,
    max_answer_len=8,
    eval_every=200,
    seed=0,
):
    rng = random.Random(seed)
    optimizer = torch.optim.AdamW(model_v.parameters(), lr=lr)

    history = []
    for step in range(1, steps + 1):
        batch = df_rl_formats.sample(batch_size, random_state=rng.randrange(10**9)).reset_index(drop=True)
        rollouts = build_batch_rollouts(model_v, batch, max_answer_len=max_answer_len, temperature=temperature)

        stats = ppo_update(
            model_v,
            rollouts,
            optimizer,
            clip_eps=0.2,
            vf_coef=0.5,
            ent_coef=0.01,
            ppo_epochs=2,
        )
        history.append(stats)

        if step % 50 == 0:
            print(f"[step {step:5d}] reward={stats['batch_reward']:.3f} "
                  f"loss={stats['loss']:.4f} pol={stats['policy_loss']:.4f} "
                  f"vf={stats['value_loss']:.4f} ent={stats['entropy']:.4f}")

        if step % eval_every == 0:
            print("PPO eval:")
            report_split("valid", df_valid, max_items=2000)
            report_split("test_seen", df_test_seen, max_items=5000)
            report_split("test_unseen_numbers", df_test_unseen_numbers, max_items=5000)
            report_split("test_unseen_formats", df_test_unseen_formats, max_items=5000)

    return history

In [208]:
history_ppo = train_ppo_formats(
    model_v=model_rl,
    df_rl_formats=df_rl_formats,
    steps=1000,
    batch_size=64,
    lr=3e-6,
    temperature=1.0,
    max_answer_len=8,
    eval_every=200,
    seed=0,
)

os.makedirs("checkpoints_rl", exist_ok=True)
torch.save(model_rl.state_dict(), "checkpoints_rl/ppo_formats_last.pt")
print("Saved: checkpoints_rl/ppo_formats_last.pt")

[step    50] reward=0.203 loss=0.1380 pol=-0.0022 vf=0.2814 ent=0.1609
[step   100] reward=0.156 loss=0.1256 pol=-0.0016 vf=0.2570 ent=0.2179
[step   150] reward=0.219 loss=0.1464 pol=-0.0020 vf=0.2996 ent=0.2416
[step   200] reward=0.219 loss=0.1180 pol=-0.0018 vf=0.2422 ent=0.2174
PPO eval:
valid: overall=0.9885 add=0.9891 sub=0.9879 by_len_a={1: 0.9960629921259843, 2: 0.9979757085020243, 3: 0.9857142857142858, 4: 0.980072463768116}
test_seen: overall=0.9898 add=0.9912 sub=0.9885 by_len_a={1: 0.9963702359346642, 2: 0.9976359338061466, 3: 0.9915349887133182, 4: 0.9779829545454546}
test_unseen_numbers: overall=0.9912 add=0.9957 sub=0.9865 by_len_a={1: 0.9983739837398374, 2: 0.9977761304670126, 3: 0.9856028794241152, 4: 0.9883126369612856}
test_unseen_formats: overall=0.2188 add=0.2198 sub=0.2178 by_len_a={1: 0.371900826446281, 2: 0.21069182389937108, 3: 0.19503750721292556, 4: 0.18920863309352517}
[step   250] reward=0.250 loss=0.1046 pol=-0.0031 vf=0.2170 ent=0.2382
[step   300] rewar

### Descussion:

In this PPO experiment, we took our already-strong supervised TinyTransformer policy and tried to fine-tune it with PPO on a subset of RL data focused on “unseen prompt formats,” using a sparse reward signal tied to correctness. The training logs show PPO is updating (non-zero policy/value/entropy terms), but the reward stays low (~0.10–0.25) and the key metric test_unseen_formats improves only marginally (≈0.219 → ≈0.227), while performance on the easier distributions (valid/seen/unseen_numbers) slowly degrades—classic policy drift/catastrophic forgetting caused by an objective that’s too sparse/brittle and not sufficiently anchored to the supervised behavior. To improve it, we’ll redesign PPO so it directly targets format generalization without sacrificing base accuracy by:

- shaping the reward (partial credit for parseable numeric answers and closeness, not only exact match)

- adding KL regularization to a frozen supervised reference policy to prevent drift

- mixing a portion of standard-format prompts into RL batches, and (4) reducing update aggressiveness (lower LR, fewer PPO epochs per batch, tighter clipping), then re-evaluate to confirm test_unseen_formats rises without harming the other splits.

# PPO second experiment:

In [209]:
import os, copy, re, math, random
import torch
import torch.nn as nn
import torch.nn.functional as F

def infer_tiny_cfg_from_state(state):
    d_model = state["embed.weight"].shape[1]
    vocab_size = state["embed.weight"].shape[0]
    max_len = state["pos.pe"].shape[0]
    layer_ids = set()
    for k in state.keys():
        m = re.match(r"blocks\.(\d+)\.", k)
        if m: layer_ids.add(int(m.group(1)))
    n_layers = max(layer_ids) + 1 if layer_ids else 0
    return dict(vocab_size=vocab_size, d_model=d_model, n_layers=n_layers, max_len=max_len)

In [210]:
def load_tiny_from_ckpt(ckpt_path, device, heads=4, dropout=0.1):
    state = torch.load(ckpt_path, map_location=device, weights_only=True)
    cfg = infer_tiny_cfg_from_state(state)
    model = TinyTransformer(
        vocab_size=cfg["vocab_size"],
        d_model=cfg["d_model"],
        n_layers=cfg["n_layers"],
        heads=heads,
        max_len=cfg["max_len"],
        dropout=dropout
    ).to(device)
    model.load_state_dict(state, strict=True)
    model.eval()
    return model, cfg

In [211]:
SUP_CKPT = "checkpoints/sup_scratch_best_by_em.pt"  # <- set yours
model_sup, sup_cfg = load_tiny_from_ckpt(SUP_CKPT, device=device, heads=4, dropout=0.1)
print("Loaded supervised model cfg:", sup_cfg)

Loaded supervised model cfg: {'vocab_size': 17, 'd_model': 256, 'n_layers': 4, 'max_len': 128}


### make an Actor-Critic wrapper for PPO:

In [212]:
class TinyPolicyValue(nn.Module):
    def __init__(self, base: TinyTransformer):
        super().__init__()
        # reuse trunk
        self.embed = base.embed
        self.pos = base.pos
        self.blocks = base.blocks
        self.ln = base.ln
        self.fc = base.fc
        self.max_len = base.max_len
        d_model = self.embed.embedding_dim
        self.v_head = nn.Linear(d_model, 1)

    def forward(self, x):
        B, T = x.size()
        if T > self.max_len:
            raise ValueError(f"T={T} exceeds max_len={self.max_len}")
        mask = torch.tril(torch.ones(T, T, device=x.device))
        h = self.embed(x)
        h = self.pos(h)
        for blk in self.blocks:
            h = blk(h, mask)
        h = self.ln(h)
        logits = self.fc(h)                 # (B,T,V)
        values = self.v_head(h).squeeze(-1) # (B,T)
        return logits, values

In [213]:
model_v = TinyPolicyValue(model_sup).to(device)
ref_policy = copy.deepcopy(model_sup).to(device).eval()  # frozen ref for KL
for p in ref_policy.parameters():
    p.requires_grad_(False)

### utilities: sampling, logprobs, reward:

In [215]:
EOS = "<eos>"  # use the same EOS string you used in supervised training

if "vocab" in globals():
    itos = {i: ch for i, ch in enumerate(vocab)}
    stoi = {ch: i for i, ch in itos.items()}
elif "itos" in globals():
    stoi = {ch: i for i, ch in itos.items()}
else:
    raise NameError("Couldn't find vocab/itos. Use Fix B to (re)create them.")

# Safety: ensure EOS exists
if EOS not in stoi:
    # append EOS to vocab
    vocab = [itos[i] for i in range(len(itos))] + [EOS]
    itos = {i: ch for i, ch in enumerate(vocab)}
    stoi = {ch: i for i, ch in itos.items()}

print("vocab_size:", len(stoi), "EOS_ID:", stoi[EOS])

vocab_size: 17 EOS_ID: 2


In [216]:
DIGITS = list("0123456789")
EOS_ID = stoi[EOS]
ALLOW_IDS = [stoi[d] for d in DIGITS] + ([stoi["-"]] if "-" in stoi else []) + [EOS_ID]
ALLOW_IDS_T = torch.tensor(ALLOW_IDS, device=device)

In [217]:
def decode_answer_tokens(tok_ids):
    s = decode_ids(tok_ids)
    s = s.replace(EOS, "").strip()
    return s

In [218]:
def parse_int(s):
    m = re.search(r"-?\d+", s.strip())
    return int(m.group(0)) if m else None

In [219]:
def reward_numeric(pred_str, gt_str):
    # strict numeric correctness (format-independent)
    gt = parse_int(str(gt_str))
    pr = parse_int(pred_str)
    return 1.0 if (gt is not None and pr == gt) else 0.0

In [220]:
@torch.no_grad()
def sample_answers_batch(model_policy: TinyTransformer, prompts, max_answer_len=8, temperature=1.0):
    # Build context ids (prompt without trailing EOS)
    ctx_list = [encode(normalize_prompt(p))[:-1] for p in prompts]
    B = len(ctx_list)
    L = max(len(x) for x in ctx_list)
    seq = torch.full((B, L + max_answer_len), EOS_ID, dtype=torch.long, device=device)
    lengths = torch.tensor([len(x) for x in ctx_list], device=device)
    for i,ids in enumerate(ctx_list):
        seq[i, :len(ids)] = torch.tensor(ids, device=device)

    finished = torch.zeros(B, dtype=torch.bool, device=device)

    for t in range(max_answer_len):
        curT = int((lengths.max()).item())
        logits = model_policy(seq[:, :curT])  # (B,curT,V)
        # take logits at last real token position for each sample
        idx = (lengths - 1).clamp(min=0)
        next_logits = logits[torch.arange(B, device=device), idx]  # (B,V)

        # restrict to allowed ids
        sub = next_logits[:, ALLOW_IDS_T] / max(temperature, 1e-6)
        probs = F.softmax(sub, dim=-1)
        samp = torch.multinomial(probs, num_samples=1).squeeze(1)          # (B,)
        next_tok = ALLOW_IDS_T[samp]                                       # (B,)

        # write token only for unfinished
        write_pos = lengths.clone()
        for i in range(B):
            if not finished[i]:
                seq[i, write_pos[i]] = next_tok[i]
                lengths[i] += 1
                if next_tok[i].item() == EOS_ID:
                    finished[i] = True

    # Extract generated answer tokens (excluding EOS)
    out_tok = []
    for i in range(B):
        start = len(ctx_list[i])
        end = int(lengths[i].item())
        gen = seq[i, start:end].tolist()
        # cut at EOS if present
        if EOS_ID in gen:
            gen = gen[:gen.index(EOS_ID)]
        out_tok.append(gen)
    return ctx_list, out_tok

In [221]:
def batch_logprob_entropy_value(model_pv: TinyPolicyValue, ctx_list, act_list):
    """
    Returns:
      logp: (B,) sum logprob of action tokens incl EOS
      ent:  (B,) sum entropy over action positions
      v0:   (B,) value at first action decision state (end of ctx)
    """
    B = len(ctx_list)
    full_inputs = []
    full_targets = []
    full_masks = []
    v_pos = []

    for ctx, act in zip(ctx_list, act_list):
        # include EOS as an action token for likelihood accounting
        act_eos = act + [EOS_ID]
        full = ctx + act_eos
        inp = full[:-1]
        tgt = full[1:]

        action_start = len(ctx) - 1                 # position predicting first action token
        mask = [0]*len(tgt)
        for k in range(action_start, action_start + len(act_eos)):
            if 0 <= k < len(mask):
                mask[k] = 1

        full_inputs.append(inp)
        full_targets.append(tgt)
        full_masks.append(mask)
        v_pos.append(action_start)

    T = max(len(x) for x in full_inputs)
    X = torch.full((B, T), EOS_ID, dtype=torch.long, device=device)
    Y = torch.full((B, T), EOS_ID, dtype=torch.long, device=device)
    M = torch.zeros((B, T), dtype=torch.float32, device=device)

    for i in range(B):
        li = len(full_inputs[i])
        X[i, :li] = torch.tensor(full_inputs[i], device=device)
        Y[i, :li] = torch.tensor(full_targets[i], device=device)
        M[i, :li] = torch.tensor(full_masks[i], device=device)

    logits, values = model_pv(X)                 # logits: (B,T,V), values: (B,T)
    log_probs = F.log_softmax(logits, dim=-1)    # (B,T,V)
    lp_tok = log_probs.gather(-1, Y.unsqueeze(-1)).squeeze(-1)  # (B,T)
    logp = (lp_tok * M).sum(dim=1)

    probs = log_probs.exp()
    ent_tok = -(probs * log_probs).sum(dim=-1)   # (B,T)
    ent = (ent_tok * M).sum(dim=1)

    v0 = torch.stack([values[i, v_pos[i]] for i in range(B)], dim=0)
    return logp, ent, v0

### PPO that targets formats without wrecking the base skill:

In [224]:
import os
os.makedirs("artifacts", exist_ok=True)

df_train.to_csv("artifacts/df_train.csv", index=False)
df_valid.to_csv("artifacts/df_valid.csv", index=False)
df_rl.to_csv("artifacts/df_rl.csv", index=False)

# If you also have format-specific RL subset:
if "df_rl_formats" in globals():
    df_rl_formats.to_csv("artifacts/df_rl_formats.csv", index=False)

print("Saved data splits to artifacts/*.csv")

Saved data splits to artifacts/*.csv


In [225]:
import json, os
os.makedirs("artifacts", exist_ok=True)

vocab_pack = {
    "EOS": EOS,
    "stoi": stoi,
    "itos": {str(k): v for k, v in itos.items()},  # json keys must be str
}
with open("artifacts/vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab_pack, f, ensure_ascii=False, indent=2)

print("Saved vocab to artifacts/vocab.json")

Saved vocab to artifacts/vocab.json


In [222]:
def report_split_model(name, df, model_policy, max_items=2000):
    em, add, sub, bylen = exact_match_accuracy(model_policy, df, max_items=max_items)
    print(f"{name}: overall={em:.4f} add={add:.4f} sub={sub:.4f} by_len_a={bylen}")

In [226]:
import torch, random, os
import numpy as np

def save_rl_checkpoint(path, model, optimizer=None, step=0, extra=None):
    os.makedirs(os.path.dirname(path), exist_ok=True)

    ckpt = {
        "step": step,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict() if optimizer is not None else None,
        "rng": {
            "python": random.getstate(),
            "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        },
        "extra": extra or {},
    }
    torch.save(ckpt, path)
    print("Saved RL checkpoint:", path)

# Example usage (adjust step number if you have it):
save_rl_checkpoint("checkpoints_rl/ppo_resume.pt", model_rl, optimizer=None, step=0)

Saved RL checkpoint: checkpoints_rl/ppo_resume.pt


In [223]:
def train_ppo_formats_anchored(
    model_pv: TinyPolicyValue,
    ref_policy: TinyTransformer,
    df_formats,
    df_anchor,
    steps=1000,
    batch_size=64,
    lr=3e-6,
    temperature=1.0,
    max_answer_len=8,
    clip_eps=0.1,
    vf_coef=0.5,
    ent_coef=0.01,
    kl_coef=0.02,
    bc_coef=0.02,          # set 0.0 to disable BC anchoring
    anchor_frac=0.30,      # 30% normal prompts, 70% format prompts
    eval_every=200,
    seed=0,
):
    torch.manual_seed(seed); random.seed(seed)
    model_pv.train()
    opt = torch.optim.AdamW(model_pv.parameters(), lr=lr)

    history = []
    best_formats = -1.0
    os.makedirs("checkpoints_rl", exist_ok=True)

    for step in range(1, steps+1):
        # ----- sample batch -----
        n_anchor = int(batch_size * anchor_frac)
        n_fmt = batch_size - n_anchor

        batch_fmt = df_formats.sample(n_fmt, replace=(n_fmt > len(df_formats)), random_state=seed+step)
        batch_anc = df_anchor.sample(n_anchor, replace=(n_anchor > len(df_anchor)), random_state=seed+10_000+step)

        batch = torch.utils.data.ConcatDataset([])  # just to keep the intent obvious
        prompts = batch_fmt["prompt"].tolist() + batch_anc["prompt"].tolist()
        gts     = batch_fmt["answer"].tolist() + batch_anc["answer"].tolist()

        # ----- rollout (sample answers) -----
        ctx_list, act_list = sample_answers_batch(ref_policy, prompts, max_answer_len=max_answer_len, temperature=temperature)
        preds = [decode_answer_tokens(a) for a in act_list]
        rewards = torch.tensor([reward_numeric(p, gt) for p,gt in zip(preds, gts)], device=device, dtype=torch.float32)

        # ----- PPO losses -----
        logp_old, _, _ = batch_logprob_entropy_value(model_pv, ctx_list, act_list)
        logp_old = logp_old.detach()

        logp_new, ent, v0 = batch_logprob_entropy_value(model_pv, ctx_list, act_list)
        # KL to ref: KL(pi || ref) approx on sampled actions = (logp_new - logp_ref)
        with torch.no_grad():
            # ref logp on same actions
            # (wrap ref in TinyPolicyValue-like call without value)
            # We'll compute ref logp by building a fake PV that returns only logits
            pass

        # compute ref logp (policy only)
        def ref_logp_only(model_policy, ctx_list, act_list):
            B = len(ctx_list)
            full_inputs, full_targets, full_masks = [], [], []
            for ctx, act in zip(ctx_list, act_list):
                act_eos = act + [EOS_ID]
                full = ctx + act_eos
                inp = full[:-1]
                tgt = full[1:]
                action_start = len(ctx) - 1
                mask = [0]*len(tgt)
                for k in range(action_start, action_start + len(act_eos)):
                    if 0 <= k < len(mask):
                        mask[k] = 1
                full_inputs.append(inp); full_targets.append(tgt); full_masks.append(mask)

            T = max(len(x) for x in full_inputs)
            X = torch.full((B, T), EOS_ID, dtype=torch.long, device=device)
            Y = torch.full((B, T), EOS_ID, dtype=torch.long, device=device)
            M = torch.zeros((B, T), dtype=torch.float32, device=device)
            for i in range(B):
                li = len(full_inputs[i])
                X[i, :li] = torch.tensor(full_inputs[i], device=device)
                Y[i, :li] = torch.tensor(full_targets[i], device=device)
                M[i, :li] = torch.tensor(full_masks[i], device=device)

            logits = model_policy(X)
            log_probs = F.log_softmax(logits, dim=-1)
            lp_tok = log_probs.gather(-1, Y.unsqueeze(-1)).squeeze(-1)
            return (lp_tok * M).sum(dim=1)

        with torch.no_grad():
            logp_ref = ref_logp_only(ref_policy, ctx_list, act_list)

        kl = (logp_new - logp_ref)  # (B,) on sampled trajectories

        adv = (rewards - v0.detach())
        ratio = torch.exp(logp_new - logp_old)
        unclipped = ratio * adv
        clipped = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * adv
        pol_loss = -torch.mean(torch.minimum(unclipped, clipped))

        vf_loss = F.mse_loss(v0, rewards)
        ent_loss = -torch.mean(ent)
        kl_loss = torch.mean(kl)

        # ----- optional BC anchoring on anchor subset -----
        # (forces the model not to drift; helps keep seen/unseen_numbers high)
        bc_loss = torch.tensor(0.0, device=device)
        if bc_coef > 0 and n_anchor > 0:
            anc_prompts = prompts[-n_anchor:]
            anc_gts = [str(x).strip() for x in gts[-n_anchor:]]
            # teacher force target = answer digits + EOS
            ctx_a = [encode(normalize_prompt(p))[:-1] for p in anc_prompts]
            act_a = [[stoi[ch] for ch in gt if ch in stoi] for gt in anc_gts]  # digits/'-'
            # compute NLL of GT actions (include EOS)
            logp_gt, _, _ = batch_logprob_entropy_value(model_pv, ctx_a, act_a)
            bc_loss = -logp_gt.mean()

        loss = pol_loss + vf_coef * vf_loss + ent_coef * ent_loss + kl_coef * kl_loss + bc_coef * bc_loss

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_pv.parameters(), 1.0)
        opt.step()

        # ----- logging -----
        if step % 50 == 0:
            history.append(dict(
                step=step,
                reward=float(rewards.mean().item()),
                loss=float(loss.item()),
                pol=float(pol_loss.item()),
                vf=float(vf_loss.item()),
                ent=float(ent.mean().item()),
                kl=float(kl.mean().item()),
                bc=float(bc_loss.item()) if torch.is_tensor(bc_loss) else float(bc_loss),
            ))
            print(f"[step {step:5d}] reward={history[-1]['reward']:.3f} "
                  f"loss={history[-1]['loss']:.4f} pol={history[-1]['pol']:.4f} "
                  f"vf={history[-1]['vf']:.4f} ent={history[-1]['ent']:.4f} "
                  f"kl={history[-1]['kl']:.4f} bc={history[-1]['bc']:.4f}")

        if step % eval_every == 0:
            print("PPO eval:")
            model_pv.eval()
            # policy for exact_match_accuracy expects a TinyTransformer-like forward -> logits
            # so we create a thin lambda-like wrapper:
            class _PolicyOnly(nn.Module):
                def __init__(self, pv): super().__init__(); self.pv=pv
                def forward(self, x): return self.pv(x)[0]
            policy_only = _PolicyOnly(model_pv).to(device)

            report_split_model("valid", df_valid, policy_only, max_items=2000)
            report_split_model("test_seen", df_test_seen, policy_only, max_items=5000)
            report_split_model("test_unseen_numbers", df_test_unseen_numbers, policy_only, max_items=5000)
            report_split_model("test_unseen_formats", df_test_unseen_formats, policy_only, max_items=5000)

            # keep best by unseen_formats
            em_f, *_ = exact_match_accuracy(policy_only, df_test_unseen_formats, max_items=5000)
            if em_f > best_formats:
                best_formats = em_f
                torch.save(model_pv.state_dict(), "checkpoints_rl/ppo_formats_best.pt")
                print(f"  ✅ Saved new BEST formats EM={best_formats:.4f} -> checkpoints_rl/ppo_formats_best.pt")

            model_pv.train()

    torch.save(model_pv.state_dict(), "checkpoints_rl/ppo_formats_last.pt")
    print("Saved: checkpoints_rl/ppo_formats_last.pt")
    return history

In [ ]:
history_ppo2 = train_ppo_formats_anchored(
    model_pv=model_v,
    ref_policy=ref_policy,
    df_formats=df_rl_formats,
    df_anchor=df_train,      # <-- anchor to prevent forgetting
    steps=1500,
    batch_size=64,
    lr=3e-6,
    temperature=1.0,
    max_answer_len=8,
    clip_eps=0.1,
    vf_coef=0.5,
    ent_coef=0.01,
    kl_coef=0.02,
    bc_coef=0.02,
    anchor_frac=0.30,
    eval_every=200,
    seed=0,
)

In [1]:
import os, shutil

os.makedirs("final_artifacts", exist_ok=True)

# Update these names to whatever you actually have on disk
CKPTS = [
    "checkpoints/sup_scratch_best_by_em.pt",          # supervised TinyTransformer
    "checkpoints_rl/reinforce_baseline_last.pt",      # RL REINFORCE
    "checkpoints_rl/ppo_formats_last.pt",             # RL PPO (if you have it)
]

for p in CKPTS:
    if os.path.exists(p):
        shutil.copy(p, "final_artifacts/" + os.path.basename(p))
        print("Saved copy:", p, "->", "final_artifacts/" + os.path.basename(p))
    else:
        print("Missing:", p)

Saved copy: checkpoints/sup_scratch_best_by_em.pt -> final_artifacts/sup_scratch_best_by_em.pt
Saved copy: checkpoints_rl/reinforce_baseline_last.pt -> final_artifacts/reinforce_baseline_last.pt
Saved copy: checkpoints_rl/ppo_formats_last.pt -> final_artifacts/ppo_formats_last.pt


# Final Evaluation:

I've restarted the kernal so I have to load everything for the evaluation

In [23]:
import os, re, time
import numpy as np
import pandas as pd
import torch
import os
import pandas as pd

In [24]:
ROOT = os.getcwd()  # your CWD already points to TP2 folder
DATA_DIR = os.path.join(ROOT, "data_task1")
ART_DIR  = os.path.join(ROOT, "artifacts")

In [25]:
def load_csv(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")
    df = pd.read_csv(path)

    df.columns = [str(c) for c in df.columns]
    if "prompt" in df.columns:
        df["prompt"] = df["prompt"].astype(str)
    if "answer" in df.columns:
        df["answer"] = df["answer"].astype(str)
    return df

In [26]:
# ---- Supervised pretraining splits ----
df_train = load_csv(os.path.join(DATA_DIR, "pretraining_train.csv"))
df_valid = load_csv(os.path.join(DATA_DIR, "pretraining_valid.csv"))

In [27]:
# ---- Evaluation splits ----
df_test_seen           = load_csv(os.path.join(DATA_DIR, "test_seen.csv"))
df_test_unseen_numbers = load_csv(os.path.join(DATA_DIR, "test_unseen_numbers.csv"))
df_test_unseen_formats = load_csv(os.path.join(DATA_DIR, "test_unseen_formats.csv"))

In [28]:
df_rl_path = os.path.join(ART_DIR, "df_rl.csv")
df_rl_formats_path = os.path.join(ART_DIR, "df_rl_formats.csv")

if os.path.exists(df_rl_path):
    df_rl = load_csv(df_rl_path)
else:
    df_rl = load_csv(os.path.join(DATA_DIR, "rl_prompts.csv"))

if os.path.exists(df_rl_formats_path):
    df_rl_formats = load_csv(df_rl_formats_path)
else:

    df_rl_formats = df_rl.copy()

df_all = pd.concat(
    [df_train, df_valid, df_test_seen, df_test_unseen_numbers, df_test_unseen_formats],
    ignore_index=True
)

print("✅ Loaded:")
print("  train:", df_train.shape)
print("  valid:", df_valid.shape)
print("  test_seen:", df_test_seen.shape)
print("  test_unseen_numbers:", df_test_unseen_numbers.shape)
print("  test_unseen_formats:", df_test_unseen_formats.shape)
print("  rl:", df_rl.shape)
print("  rl_formats:", df_rl_formats.shape)
print("  df_all (train+valid+tests):", df_all.shape)

✅ Loaded:
  train: (200000, 9)
  valid: (10000, 9)
  test_seen: (20000, 9)
  test_unseen_numbers: (20000, 9)
  test_unseen_formats: (20000, 9)
  rl: (50000, 9)
  rl_formats: (50000, 2)
  df_all (train+valid+tests): (270000, 9)


sanity: required columns

In [29]:
needed = {"prompt", "answer"}
for name, df in [("train", df_train), ("valid", df_valid), ("test_seen", df_test_seen),
                 ("test_unseen_numbers", df_test_unseen_numbers), ("test_unseen_formats", df_test_unseen_formats)]:
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")

# quick look
display(df_valid.head(3))

,prompt,answer,op,a,b,fmt_id,len_a,len_b,len_ans
0,2804 - 51 =,2753,-,2804,51,0,4,2,4
1,89 + 47 =,136,+,89,47,0,2,2,3
2,5 + 5 =,10,+,5,5,0,1,1,2


In [30]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [31]:
ROOT = os.getcwd()
DATA_DIR = os.path.join(ROOT, "data_task1")
ART_DIR  = os.path.join(ROOT, "artifacts")
os.makedirs("results", exist_ok=True)

In [32]:
def load_csv(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")
    df = pd.read_csv(path)
    df["prompt"] = df["prompt"].astype(str)
    df["answer"] = df["answer"].astype(str)
    return df.reset_index(drop=True)

## Supervised + tests:

In [33]:
df_train = load_csv(os.path.join(DATA_DIR, "pretraining_train.csv"))
df_valid = load_csv(os.path.join(DATA_DIR, "pretraining_valid.csv"))
df_test_seen = load_csv(os.path.join(DATA_DIR, "test_seen.csv"))
df_test_unseen_numbers = load_csv(os.path.join(DATA_DIR, "test_unseen_numbers.csv"))
df_test_unseen_formats = load_csv(os.path.join(DATA_DIR, "test_unseen_formats.csv"))

In [34]:
df_rl = load_csv(os.path.join(ART_DIR, "df_rl.csv")) if os.path.exists(os.path.join(ART_DIR, "df_rl.csv")) else None
df_rl_formats = load_csv(os.path.join(ART_DIR, "df_rl_formats.csv")) if os.path.exists(os.path.join(ART_DIR, "df_rl_formats.csv")) else None

In [35]:
print("Loaded:")
print(" train:", df_train.shape, " valid:", df_valid.shape)
print(" test_seen:", df_test_seen.shape)
print(" test_unseen_numbers:", df_test_unseen_numbers.shape)
print(" test_unseen_formats:", df_test_unseen_formats.shape)
if df_rl is not None: print(" df_rl:", df_rl.shape)
if df_rl_formats is not None: print(" df_rl_formats:", df_rl_formats.shape)

Loaded:
 train: (200000, 9)  valid: (10000, 9)
 test_seen: (20000, 9)
 test_unseen_numbers: (20000, 9)
 test_unseen_formats: (20000, 9)
 df_rl: (50000, 9)
 df_rl_formats: (50000, 2)


In [36]:
SPLITS = {
    "valid": df_valid,
    "test_seen": df_test_seen,
    "test_unseen_numbers": df_test_unseen_numbers,
    "test_unseen_formats": df_test_unseen_formats,
}

In [37]:
TOKENS = list("0123456789+-= ") + ["<pad>", "<bos>", "<eos>"]
stoi = {t:i for i,t in enumerate(TOKENS)}
itos = {i:t for t,i in stoi.items()}
PAD, BOS, EOS = "<pad>", "<bos>", "<eos>"
PAD_ID, BOS_ID, EOS_ID = stoi[PAD], stoi[BOS], stoi[EOS]

ALLOW_TOKENS = list("0123456789") + (["-"] if "-" in stoi else []) + [EOS]
ALLOW_IDS = torch.tensor([stoi[t] for t in ALLOW_TOKENS], device=device)

In [38]:
def encode_text(s: str):
    # any unknown char becomes space (safe)
    ids = [stoi.get(ch, stoi[" "]) for ch in s]
    return ids

def normalize_ans(x: str) -> str:
    m = re.search(r"-?\d+", str(x).strip())
    return m.group(0) if m else ""

def infer_op(prompt: str) -> str:
    if "+" in prompt: return "+"
    if "-" in prompt: return "-"
    return "?"

In [39]:
@torch.no_grad()
def generate_answer_tiny(model, prompt: str, max_answer_len=8, temperature=0.0):
    """
    Autoregressive decode constrained to digits/'-'/'<eos>'.
    Returns normalized integer string.
    """
    model.eval()

    # input: <bos> + prompt chars
    x = [BOS_ID] + encode_text(prompt)
    x = torch.tensor(x, dtype=torch.long, device=device).unsqueeze(0)  # (1, T)

    out_ids = []
    for _ in range(max_answer_len):
        logits = model(x)  # expected shape (1, T, vocab)
        last = logits[0, -1, :]  # (vocab,)

        # mask to allowed ids only
        masked = torch.full_like(last, -1e9)
        masked[ALLOW_IDS] = last[ALLOW_IDS]

        if temperature and temperature > 0:
            probs = torch.softmax(masked / float(temperature), dim=-1)
            next_id = torch.multinomial(probs, 1).item()
        else:
            next_id = int(torch.argmax(masked).item())

        if next_id == EOS_ID:
            break

        out_ids.append(next_id)
        x = torch.cat([x, torch.tensor([[next_id]], device=device)], dim=1)

    text = "".join(itos[i] for i in out_ids)
    return normalize_ans(text)

In [40]:
def report_split(name, df, model, max_items=2000, max_answer_len=8, temperature=0.0):
    df_eval = df.sample(n=min(max_items, len(df)), random_state=0).reset_index(drop=True)

    ok_all = 0
    ok_add = 0; n_add = 0
    ok_sub = 0; n_sub = 0
    by_len = {1:[0,0], 2:[0,0], 3:[0,0], 4:[0,0]}

    for _, r in df_eval.iterrows():
        p = r["prompt"]
        gt = normalize_ans(r["answer"])
        pred = generate_answer_tiny(model, p, max_answer_len=max_answer_len, temperature=temperature)

        ok = (pred == gt) and (gt != "")
        ok_all += int(ok)

        op = infer_op(p)
        if op == "+":
            n_add += 1; ok_add += int(ok)
        elif op == "-":
            n_sub += 1; ok_sub += int(ok)

        la = int(r["len_a"]) if "len_a" in r and not pd.isna(r["len_a"]) else -1
        if la in by_len:
            by_len[la][1] += 1
            by_len[la][0] += int(ok)

    overall = ok_all / len(df_eval)
    add = ok_add / n_add if n_add else 0.0
    sub = ok_sub / n_sub if n_sub else 0.0
    by_len_a = {k: (by_len[k][0] / by_len[k][1] if by_len[k][1] else 0.0) for k in sorted(by_len)}

    print(f"{name}: overall={overall:.4f} add={add:.4f} sub={sub:.4f} by_len_a={by_len_a}")
    return {"split": name, "overall": overall, "add": add, "sub": sub, "by_len_a": str(by_len_a), "n": len(df_eval)}

In [41]:
def eval_model_on_splits(model, name, max_items_valid=2000, max_items_test=5000, max_answer_len=8, temperature=0.0):
    rows = []
    rows.append(report_split("valid", SPLITS["valid"], model, max_items=max_items_valid, max_answer_len=max_answer_len, temperature=temperature))
    rows.append(report_split("test_seen", SPLITS["test_seen"], model, max_items=max_items_test, max_answer_len=max_answer_len, temperature=temperature))
    rows.append(report_split("test_unseen_numbers", SPLITS["test_unseen_numbers"], model, max_items=max_items_test, max_answer_len=max_answer_len, temperature=temperature))
    rows.append(report_split("test_unseen_formats", SPLITS["test_unseen_formats"], model, max_items=max_items_test, max_answer_len=max_answer_len, temperature=temperature))

    out = pd.DataFrame(rows)
    ts = time.strftime("%Y%m%d_%H%M%S")
    out_path = os.path.join("results", f"eval_{name}_{ts}.csv")
    out.to_csv(out_path, index=False)
    print("✅ Saved:", out_path)
    return out

In [105]:
if "TinyTransformer" not in globals():
    raise NameError("TinyTransformer class is not defined. Run the cell that defines TinyTransformer first.")

# IMPORTANT: Your checkpoint indicates d_model=256, max_len=128, n_layers=4 (from your earlier error logs).
vocab_size = 17
model_sup = TinyTransformer(vocab_size=vocab_size, d_model=256, n_layers=4, heads=4, max_len=128).to(device)

def safe_torch_load(path):
    # avoid future warning + safer loading when possible
    try:
        return torch.load(path, map_location=device, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=device)

In [106]:
SUP_CKPT = os.path.join("checkpoints", "sup_scratch_best_by_em.pt")
if not os.path.exists(SUP_CKPT):
    # fallback: pick a likely supervised checkpoint if filename differs
    import glob
    cands = sorted(glob.glob(os.path.join("checkpoints", "*.pt")))
    if len(cands) == 0:
        raise FileNotFoundError("No supervised checkpoint found in ./checkpoints/")
    SUP_CKPT = cands[0]
    print("⚠️ Using fallback supervised checkpoint:", SUP_CKPT)

In [107]:
state = safe_torch_load(SUP_CKPT)
model_sup.load_state_dict(state, strict=True)
print("✅ Loaded supervised:", SUP_CKPT)

print("\n=== EVAL: supervised model ===")
sup_df = eval_model_on_splits(model_sup, "tiny_supervised")

✅ Loaded supervised: checkpoints\sup_scratch_best_by_em.pt

=== EVAL: supervised model ===
valid: overall=0.0005 add=0.0000 sub=0.0010 by_len_a={1: 0.0, 2: 0.0020242914979757085, 3: 0.0, 4: 0.0}
test_seen: overall=0.0006 add=0.0004 sub=0.0008 by_len_a={1: 0.0, 2: 0.002364066193853428, 3: 0.0, 4: 0.0}
test_unseen_numbers: overall=0.0004 add=0.0004 sub=0.0004 by_len_a={1: 0.0, 2: 0.0007412898443291327, 3: 0.0005998800239952009, 4: 0.0}
test_unseen_formats: overall=0.0006 add=0.0000 sub=0.0012 by_len_a={1: 0.001652892561983471, 2: 0.0015723270440251573, 3: 0.0, 4: 0.0}
✅ Saved: results\eval_tiny_supervised_20251218_214607.csv


In [108]:
RL_CKPT = os.path.join("checkpoints_rl", "reinforce_baseline_last.pt")
if os.path.exists(RL_CKPT):
    model_rl = TinyTransformer(vocab_size=vocab_size, d_model=256, n_layers=4, heads=4, max_len=128).to(device)
    model_rl.load_state_dict(safe_torch_load(RL_CKPT), strict=True)
    print("\n✅ Loaded RL:", RL_CKPT)
    print("\n=== EVAL: after RL (reinforce_baseline_last) ===")
    rl_df = eval_model_on_splits(model_rl, "tiny_after_reinforce")
else:
    print("\nℹ️ RL checkpoint not found:", RL_CKPT)
    print("   If you want PPO evaluation instead, set RL_CKPT to checkpoints_rl/ppo_formats_last.pt")


✅ Loaded RL: checkpoints_rl\reinforce_baseline_last.pt

=== EVAL: after RL (reinforce_baseline_last) ===
valid: overall=0.0005 add=0.0000 sub=0.0010 by_len_a={1: 0.0, 2: 0.0020242914979757085, 3: 0.0, 4: 0.0}
test_seen: overall=0.0006 add=0.0004 sub=0.0008 by_len_a={1: 0.0, 2: 0.002364066193853428, 3: 0.0, 4: 0.0}
test_unseen_numbers: overall=0.0004 add=0.0004 sub=0.0004 by_len_a={1: 0.0, 2: 0.0007412898443291327, 3: 0.0005998800239952009, 4: 0.0}
test_unseen_formats: overall=0.0006 add=0.0000 sub=0.0012 by_len_a={1: 0.001652892561983471, 2: 0.0015723270440251573, 3: 0.0, 4: 0.0}
✅ Saved: results\eval_tiny_after_reinforce_20251218_214944.csv
